# Sistema de Recomendação de Produtos Financeiros
## Case Técnico — Cientista de Dados Sênior | Itaú

### Problema de Negócio
O Itaú exibe **20 produtos** em carrossel horizontal no app. **Apenas as 5 primeiras posições são visíveis sem scroll**, tornando a ordenação crítica para conversão. A ordem atual é estática por segmento.

**Objetivo:** construir um sistema de recomendação personalizado que maximize contratação e receita, priorizando os produtos mais relevantes para cada cliente nas primeiras posições.

### Estrutura
1. EDA — entendimento da base e padrões de comportamento
2. Feature Engineering — representações ricas com correção de viés de posição
3. Modelagem — LightGBM (classificação supervisionada) + Filtragem Colaborativa (SVD matricial)
4. Avaliação — Precision@5, NDCG@5 e métricas secundárias vs. 3 baselines
5. Recomendações finais — `recomendacoes.csv` para toda a base
6. Proposta de Produção


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from itertools import combinations
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings, os, random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})
sns.set_style('whitegrid')
sns.set_palette('Set2')
warnings.filterwarnings('ignore')

DATA_DIR   = 'data'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Split temporal: 21 safras treino (202401-202509) | 3 safras teste (202510-202512)
# Justificativa: 3 meses = ~60k interações, volume estatisticamente robusto para
# avaliar Precision@5 e NDCG@5; deixa 87.5% dos dados para treino.
TRAIN_CUTOFF    = 202510
TEST_SAFRAS     = [202510, 202511, 202512]
TEST_START_DATE = pd.Timestamp('2025-10-01')
K_TOP  = 5
K_VALS = [5, 10, 20]


## 1. Carregamento dos Dados

| Tabela | Descrição | Dimensão esperada |
|--------|-----------|-------------------|
| `clientes.csv` | Perfil socioeconômico e comportamental | ~50 k × 23 |
| `interacoes.csv` | Exibições no carrossel (cliques + contratos) | ~500 k × 10 |
| `produtos.csv` | Metadados financeiros dos 20 produtos | 20 × 6 |
| `contratos_ativos.csv` | Histórico de contratos com status | ~62 k × 5 |

**Atenção aos 3 campos de canal distintos** — `canal_preferencial` (clientes), `canal` (interacoes) e `canal_contratacao` (contratos) têm domínios diferentes e não devem ser unificados sem mapeamento explícito.


In [ ]:
clientes   = pd.read_csv(f'{DATA_DIR}/clientes.csv')
interacoes = pd.read_csv(f'{DATA_DIR}/interacoes.csv', parse_dates=['timestamp'])
produtos   = pd.read_csv(f'{DATA_DIR}/produtos.csv')
contratos  = pd.read_csv(f'{DATA_DIR}/contratos_ativos.csv', parse_dates=['data_contratacao'])

for name, df in [('Clientes', clientes), ('Interações', interacoes),
                 ('Produtos', produtos), ('Contratos', contratos)]:
    print(f'{name}: {df.shape[0]:,} linhas × {df.shape[1]} colunas')

print(f'\nSafras disponíveis: {sorted(interacoes.safra.unique())}')
print(f'Split: treino < {TRAIN_CUTOFF} | teste {TEST_SAFRAS}')
print(f'\nStatus dos contratos:\n{contratos.status.value_counts()}')
print(f'\nCanais (interacoes): {interacoes.canal.unique()}')
print(f'Canal preferencial (clientes): {clientes.canal_preferencial.unique()}')
print(f'Canal contratação (contratos): {contratos.canal_contratacao.unique()}')


In [ ]:
clientes

In [ ]:
interacoes

In [ ]:
produtos

In [ ]:
contratos

## 2. Análise Exploratória dos Dados (EDA)

Objetivo: entender o perfil da base, padrões de comportamento e vieses estruturais que impactam a modelagem.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

colors = {'basico': '#5B9BD5', 'intermediario': '#70AD47', 'premium': '#FFC000'}

# Distribuição por segmento
seg_counts = clientes['segmento'].value_counts().reindex(['basico','intermediario','premium'])
bars = axes[0,0].bar(seg_counts.index, seg_counts.values,
                      color=[colors[s] for s in seg_counts.index])
axes[0,0].set_title('Distribuição por Segmento')
axes[0,0].set_ylabel('Qtd. Clientes')
for bar, v in zip(bars, seg_counts.values):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, v + 300,
                   f'{v:,}\n({v/len(clientes):.1%})', ha='center', fontsize=9)

# Faixa etária por segmento
clientes['faixa_etaria'] = pd.cut(clientes['idade'],
    bins=[17,25,35,45,55,65,80], labels=['18-25','26-35','36-45','46-55','56-65','66+'])
faixa_seg = (clientes.groupby(['faixa_etaria','segmento'], observed=True)
             .size().unstack(fill_value=0))
faixa_seg.plot(kind='bar', ax=axes[0,1], stacked=True,
               color=[colors[c] for c in faixa_seg.columns])
axes[0,1].set_title('Faixa Etária por Segmento')
axes[0,1].set_xlabel('')
axes[0,1].tick_params(axis='x', rotation=30)
axes[0,1].legend(title='Segmento', fontsize=8)

# Score de crédito por segmento
for seg, color in colors.items():
    data = clientes.loc[clientes['segmento']==seg, 'score_credito'].dropna()
    axes[0,2].hist(data, bins=25, alpha=0.6, label=seg, color=color, density=True)
axes[0,2].set_title('Score de Crédito por Segmento')
axes[0,2].set_xlabel('Score (0-1000)')
axes[0,2].legend()

# Renda mensal por segmento (boxplot manual para controlar a ordem)
seg_order = ['basico','intermediario','premium']
data_box = [clientes.loc[clientes['segmento']==s, 'renda_mensal'].dropna().values
             for s in seg_order]
bp = axes[1,0].boxplot(data_box, patch_artist=True,
                        medianprops=dict(color='red', linewidth=2))
for patch, color in zip(bp['boxes'], ['#5B9BD5','#70AD47','#FFC000']):
    patch.set_facecolor(color)
axes[1,0].set_xticklabels(seg_order)
axes[1,0].set_title('Renda Mensal por Segmento')
axes[1,0].set_xlabel('')
axes[1,0].set_ylabel('R$')

# Produtos ativos por segmento
prod_seg = clientes.groupby('segmento')['qtd_produtos_ativos'].mean().reindex(
    ['basico','intermediario','premium'])
b = axes[1,1].bar(prod_seg.index, prod_seg.values,
                   color=[colors[s] for s in prod_seg.index])
axes[1,1].set_title('Média de Produtos Ativos por Segmento')
axes[1,1].set_ylabel('Qtd. Média')
for bar, v in zip(b, prod_seg.values):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, v + 0.03,
                   f'{v:.2f}', ha='center')

# Canal preferencial
canal_counts = clientes['canal_preferencial'].value_counts()
axes[1,2].pie(canal_counts.values, labels=canal_counts.index,
              autopct='%1.1f%%', startangle=90,
              colors=sns.color_palette('Set2', len(canal_counts)))
axes[1,2].set_title('Canal Preferencial dos Clientes')

plt.suptitle('Perfil da Base de Clientes', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_clientes.png', bbox_inches='tight', dpi=120)
plt.show()

print('\nEstatísticas de renda por segmento:')
print(clientes.groupby('segmento')['renda_mensal']
      .agg(['median','mean','std']).round(2))


**Insight de negócio — Perfil dos clientes:**
- A base é predominantemente **básica (55%)**, o que significa que a maioria dos clientes tem renda e score mais baixos, e poucos produtos ativos — alto potencial de cross-sell.
- Clientes **premium** têm em média 4.03 produtos ativos; básicos têm 1.20 e intermediários 2.50 — a estratégia de recomendação deve ser diferenciada por segmento, priorizando cross-sell para básicos e upsell para premium.
- O **app** é o canal preferencial da maioria, validando o carrossel como ponto de ativação central.
- Score de crédito se distribui de forma distinta por segmento — é uma feature discriminante importante para modelagem.


In [ ]:
prod_stats = (interacoes.groupby('produto')
              .agg(total_views=('id_interacao','count'),
                   total_clicks=('clicou','sum'),
                   total_contracts=('contratou','sum'),
                   total_revenue=('receita_gerada','sum'))
              .reset_index())

prod_stats['ctr'] = prod_stats['total_clicks'] / prod_stats['total_views']
prod_stats['cvr'] = prod_stats['total_contracts'] / prod_stats['total_views']
prod_stats = prod_stats.merge(produtos[['produto','categoria','receita_media']], on='produto')
prod_stats_sorted = prod_stats.sort_values('total_contracts', ascending=True)

media_ctr = prod_stats['ctr'].mean()
media_cvr = prod_stats['cvr'].mean()
palette = sns.color_palette('Set2', len(prod_stats_sorted))

# --- Figura 1: CTR, CVR, Receita Total, Receita Média ---
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

axes[0,0].barh(prod_stats_sorted['produto'], prod_stats_sorted['ctr']*100, color=palette)
axes[0,0].axvline(media_ctr*100, color='red', linestyle='--',
                  label=f'Média: {media_ctr:.2%}')
axes[0,0].set_title('Taxa de Clique (CTR) por Produto')
axes[0,0].set_xlabel('CTR (%)')
axes[0,0].legend(fontsize=9)

axes[0,1].barh(prod_stats_sorted['produto'], prod_stats_sorted['cvr']*100, color=palette)
axes[0,1].axvline(media_cvr*100, color='red', linestyle='--',
                  label=f'Média: {media_cvr:.2%}')
axes[0,1].set_title('Taxa de Contratação (CVR) por Produto')
axes[0,1].set_xlabel('CVR (%)')
axes[0,1].legend(fontsize=9)

axes[1,0].barh(prod_stats_sorted['produto'], prod_stats_sorted['total_revenue']/1e3, color=palette)
axes[1,0].set_title('Receita Total Gerada (R$ mil)')
axes[1,0].set_xlabel('Receita total (R$ mil)')

# Receita média por contrato (receita_media do produtos.csv)
prod_stats_rm = prod_stats.sort_values('receita_media', ascending=True)
bar_colors_rm = ['#E53935' if v >= 200 else '#42A5F5' for v in prod_stats_rm['receita_media']]
axes[1,1].barh(prod_stats_rm['produto'], prod_stats_rm['receita_media'], color=bar_colors_rm)
axes[1,1].axvline(prod_stats['receita_media'].mean(), color='red', linestyle='--',
                  label=f'Média: R${prod_stats["receita_media"].mean():.0f}')
axes[1,1].set_title('Receita Média por Contrato — Lucratividade Unitária\n(vermelho = acima de R$200)')
axes[1,1].set_xlabel('Receita média por contrato (R$)')
axes[1,1].legend(fontsize=9)

plt.suptitle('Desempenho dos Produtos no Carrossel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_produtos.png', bbox_inches='tight', dpi=120)
plt.show()

# --- Figura 2: Scatter CTR vs CVR por quadrante ---
fig2, ax2 = plt.subplots(figsize=(10, 7))

cat_palette = {c: sns.color_palette('tab10')[i]
               for i, c in enumerate(prod_stats['categoria'].unique())}
for cat, grp in prod_stats.groupby('categoria'):
    ax2.scatter(grp['ctr']*100, grp['cvr']*100,
                color=cat_palette[cat], s=grp['receita_media']*0.8,
                label=cat, alpha=0.85, edgecolors='white', linewidths=0.5)
    for _, row in grp.iterrows():
        ax2.annotate(row['produto'], (row['ctr']*100, row['cvr']*100),
                     fontsize=7.5, ha='left', va='bottom',
                     xytext=(4, 3), textcoords='offset points')

ax2.axvline(media_ctr*100, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.axhline(media_cvr*100, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.text(media_ctr*100 + 0.2, ax2.get_ylim()[1]*0.98 if ax2.get_ylim()[1] > 0 else 1.9,
         'CTR médio', fontsize=8, color='gray', va='top')
ax2.text(ax2.get_xlim()[0] if ax2.get_xlim()[0] > 0 else 0.1,
         media_cvr*100 + 0.02, 'CVR médio', fontsize=8, color='gray')

# Labels dos quadrantes
xmax = prod_stats['ctr'].max()*100
ymax = prod_stats['cvr'].max()*100
ax2.text(media_ctr*100*1.35, ymax*0.95,
         'Alto CTR\nAlto CVR\n(estrelas)', ha='center', fontsize=9,
         color='#1565C0', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#E3F2FD', alpha=0.7))
ax2.text(media_ctr*100*0.35, ymax*0.95,
         'Baixo CTR\nAlto CVR\n(oportunidade?)', ha='center', fontsize=9,
         color='#B71C1C', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE', alpha=0.7))
ax2.text(media_ctr*100*1.35, media_cvr*100*0.3,
         'Alto CTR\nBaixo CVR\n(curiosidade sem conversão)', ha='center',
         fontsize=8.5, color='#E65100',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0', alpha=0.7))
ax2.text(media_ctr*100*0.35, media_cvr*100*0.3,
         'Baixo CTR\nBaixo CVR\n(nicho/complexo)', ha='center',
         fontsize=8.5, color='#4E342E',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#EFEBE9', alpha=0.7))

ax2.set_xlabel('CTR — Taxa de Clique (%)')
ax2.set_ylabel('CVR — Taxa de Contratação (%)')
ax2.set_title('Scatter CTR vs. CVR por Produto\n(tamanho do ponto ∝ receita_media)',
              fontsize=13, fontweight='bold')
ax2.legend(title='Categoria', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_produtos_scatter.png', bbox_inches='tight', dpi=120)
plt.show()

# --- Prints ---
print('\nTop 5 por CVR:')
print(prod_stats.nlargest(5,'cvr')[['produto','ctr','cvr','receita_media']].to_string(index=False))
print('\nTop 5 por receita média por contrato (lucratividade unitária):')
print(prod_stats.nlargest(5,'receita_media')[['produto','cvr','receita_media','total_contracts']].to_string(index=False))

print('\nProdutos com CVR acima da média mas CTR abaixo da média (candidatos a subir no ranking):')
candidatos = prod_stats[(prod_stats['cvr'] > media_cvr) & (prod_stats['ctr'] < media_ctr)]
if candidatos.empty:
    print('  Nenhum produto encontrado neste quadrante.')
else:
    print(candidatos[['produto','ctr','cvr','receita_media']].to_string(index=False))


**Insight de negócio — Produtos:**

- **O scatter CTR × CVR revela uma correlação positiva entre clique e contratação:** nenhum produto ocupa o quadrante "baixo CTR, alto CVR" — produtos mais clicados também são mais contratados. Isso indica que o comportamento de exploração (clique) e de conversão (contratação) seguem a mesma lógica de interesse, tornando o CTR um proxy confiável de demanda.

- **Dois grupos estratégicos distintos emergem:**
  - **Alto CTR + Alto CVR (9 produtos):** `credito_pessoal`, `conta_digital_plus`, `cartao_platinum`, `previdencia`, `seguro_auto`, `investimento_cdb`, `seguro_residencial`, `cheque_especial`, `pix_parcelado` — são os produtos de maior tração e devem garantir espaço nas primeiras posições para a maioria dos segmentos.
  - **Baixo CTR + Baixo CVR (10 produtos):** `consorcio_imovel`, `consorcio_auto`, `cartao_black`, `credito_consignado`, `tesouro_direto`, entre outros — produtos de nicho ou alta complexidade que exigem personalização e contexto certo para converter.

- **A maior tensão estratégica está na lucratividade unitária:** `consorcio_imovel` gera R$450 por contrato (maior receita_media do catálogo), mas tem CVR baixíssimo. Da mesma forma, `credito_consignado` rende R$180/contrato com CVR modesto. O sistema de recomendação precisa equilibrar **probabilidade de contratação** (CVR) com **valor por contrato** (receita_media) — não otimizar só um deles.

- **`credito_pessoal` é o produto mais estratégico:** melhor combinação de CVR alto (1.79%) e receita_media elevada (R$210/contrato). Deve ser prioridade no carrossel para clientes elegíveis ao crédito.


In [ ]:
pos_stats = (interacoes.groupby('posicao_exibicao')
             .agg(views=('id_interacao','count'),
                  clicks=('clicou','sum'),
                  contracts=('contratou','sum'))
             .reset_index())
pos_stats['ctr'] = pos_stats['clicks'] / pos_stats['views']
pos_stats['cvr'] = pos_stats['contracts'] / pos_stats['views']
pos_stats['propensity'] = pos_stats['ctr'] / pos_stats['ctr'].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bar_colors = ['#1565C0' if i < 5 else '#90CAF9' for i in range(20)]
axes[0].bar(pos_stats['posicao_exibicao'], pos_stats['ctr']*100, color=bar_colors)
axes[0].axvline(5.5, color='red', linestyle='--', linewidth=2, label='Limite visível sem scroll')
axes[0].set_title('CTR por Posição no Carrossel')
axes[0].set_xlabel('Posição')
axes[0].set_ylabel('CTR (%)')
axes[0].legend()
axes[0].text(2.5, pos_stats['ctr'].max()*100*0.85,
             'Visível\nsem scroll', ha='center', color='#1565C0', fontweight='bold', fontsize=10)

axes[1].plot(pos_stats['posicao_exibicao'], pos_stats['propensity'],
             'o-', color='#C62828', linewidth=2, markersize=6)
axes[1].axvline(5.5, color='gray', linestyle='--', linewidth=1.5)
axes[1].fill_between(pos_stats['posicao_exibicao'], pos_stats['propensity'],
                     alpha=0.15, color='#C62828')
axes[1].set_title('Propensidade de Clique por Posição\n(normalizada pela posição 1)')
axes[1].set_xlabel('Posição')
axes[1].set_ylabel('Propensidade relativa')
axes[1].axhline(1, color='black', linestyle=':', alpha=0.5)

plt.suptitle('Viés de Posição — Evidência Empírica', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_position_bias.png', bbox_inches='tight', dpi=120)
plt.show()

ctr1 = pos_stats.loc[pos_stats['posicao_exibicao']==1,'ctr'].values[0]
ctr20 = pos_stats.loc[pos_stats['posicao_exibicao']==20,'ctr'].values[0]
cliques_top5 = pos_stats[pos_stats['posicao_exibicao']<=5]['clicks'].sum()
cliques_total = pos_stats['clicks'].sum()
print(f'CTR posição 1 vs posição 20: {ctr1:.4f} vs {ctr20:.4f} ({ctr1/ctr20:.1f}x maior)')
print(f'Posições 1-5 concentram {cliques_top5/cliques_total:.1%} dos cliques')
print(f'\nPropensidade por posição (top 10):')
print(pos_stats[['posicao_exibicao','ctr','propensity']].head(10).to_string(index=False))


**Insight crítico — Viés de Posição:**
- As 5 primeiras posições concentram a **90.6% dos cliques**, confirmando a hipótese central do negócio de que a ordem do carrossel define o que o cliente vê e contrata.
- A CTR cai de forma acentuada conforme a posição aumenta — a posição 1 tem CTR 27.3× maior que a posição 20. Há uma queda especialmente brusca entre a posição 5 (propensity 0.55) e a posição 6 (0.22), o chamado "efeito scroll" — produtos além da posição 5 perdem mais de metade da atenção do cliente de forma abrupta.
- **Implicação para modelagem:** interações em posições baixas são sub-amostradas em termos de interesse real do cliente. Se treinarmos ingenuamente, o modelo aprende a recomendar os produtos mais populares (que sempre aparecem no topo), em vez dos mais relevantes para cada cliente.
- **Solução:** aplicar **Inverse Propensity Weighting (IPW)** no treino — interações em posições piores recebem peso maior, compensando o viés de exposição.

In [ ]:
canal_stats = (interacoes.groupby('canal')
               .agg(views=('id_interacao','count'),
                    clicks=('clicou','sum'),
                    contracts=('contratou','sum'))
               .reset_index())
canal_stats['ctr'] = canal_stats['clicks'] / canal_stats['views']
canal_stats['cvr'] = canal_stats['contracts'] / canal_stats['views']
canal_stats = canal_stats.sort_values('ctr', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(canal_stats))
w = 0.35
axes[0].bar([i-w/2 for i in x], canal_stats['ctr']*100, w, label='CTR', color='#42A5F5')
axes[0].bar([i+w/2 for i in x], canal_stats['cvr']*100, w, label='CVR', color='#66BB6A')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(canal_stats['canal'], rotation=20, ha='right')
axes[0].set_title('CTR e CVR por Canal de Interação\n(interacoes.csv — campo "canal")')
axes[0].set_ylabel('Taxa (%)')
axes[0].legend()

canal_cont = contratos.groupby('canal_contratacao').size().reset_index(name='total')
canal_cont = canal_cont.sort_values('total', ascending=False)
axes[1].bar(canal_cont['canal_contratacao'], canal_cont['total'],
            color=sns.color_palette('Set2', len(canal_cont)))
axes[1].set_title('Contratos por Canal de Contratação\n(contratos_ativos.csv — campo "canal_contratacao")')
axes[1].set_ylabel('Total de Contratos')
for i, row in enumerate(canal_cont.itertuples()):
    axes[1].text(i, row.total + 200, f'{row.total:,}\n({row.total/canal_cont["total"].sum():.1%})',
                 ha='center', fontsize=9)

plt.suptitle('Análise por Canal — 3 Campos Distintos, Semânticas Diferentes',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_canais.png', bbox_inches='tight', dpi=120)
plt.show()


**Insight — Canais:**
- `app_busca` se destaca com o maior CTR (~22%) e o maior CVR (~4.5%) — clientes que chegam via busca já têm intenção ativa, o que explica a alta taxa de conversão. É o contexto de maior qualidade de engajamento no carrossel.
- `app_home` tem CTR intermediário (~9%) mas CVR próximo de zero — clientes navegam pela home mas raramente contratam diretamente, sugerindo que o carrossel na home funciona mais como canal de descoberta do que de conversão imediata.
- `push_notification` tem o menor CTR (~3.5%) e CVR quase zero — notificações push geram pouco engajamento qualificado, indicando que a segmentação das campanhas de push pode estar pouco personalizada.
- A maioria dos contratos efetivos acontece no app (45.2%), seguido de web (24.8%) e agência (20.1%) — o digital responde por 70% das contratações, mas a agência ainda é relevante, possivelmente para produtos mais complexos como consórcio e previdência.

In [ ]:
ativos = contratos[contratos['status']=='ativo']
client_prod_list = ativos.groupby('id_cliente')['produto'].apply(list)

all_prod_names = produtos['produto'].tolist()
co_matrix = pd.DataFrame(0, index=all_prod_names, columns=all_prod_names)

for prods in client_prod_list:
    unique_prods = list(set(prods))
    for p1, p2 in combinations(unique_prods, 2):
        if p1 in co_matrix.index and p2 in co_matrix.columns:
            co_matrix.loc[p1, p2] += 1
            co_matrix.loc[p2, p1] += 1

n_clients_active = ativos['id_cliente'].nunique()
co_pct = co_matrix / n_clients_active * 100

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.eye(len(co_pct), dtype=bool)
sns.heatmap(co_pct, mask=mask, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.3, ax=ax,
            cbar_kws={'label': '% da base com ambos os produtos ativos'})
ax.set_title('Matriz de Co-Contratação de Produtos\n(% da base com ambos os produtos ativos)',
             fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_co_contratacao.png', bbox_inches='tight', dpi=120)
plt.show()

co_long = co_matrix.stack().reset_index()
co_long.columns = ['prod_a','prod_b','co_count']
co_long = (co_long[co_long['prod_a'] < co_long['prod_b']]
           .sort_values('co_count', ascending=False))
print('Top 10 pares mais co-contratados:')
print(co_long.head(10).to_string(index=False))


**Insight — Co-Contratação:**
- O par mais co-contratado da base é `conta_digital_plus + credito_pessoal` com 4.3% da base tendo ambos ativos simultaneamente — sugere que clientes que abrem conta digital frequentemente buscam crédito na sequência, um padrão de jornada de onboarding que o modelo deve explorar.
- Crédito coexiste com frequência: `cheque_especial + conta_digital_plus` (2.6%), `cheque_especial + credito_pessoal` (2.5%) são pares comuns — produtos de crédito se complementam na carteira dos clientes.
- `cartao_black` tem a menor co-contratação de todos os produtos — valores próximos de 0.0 em toda a linha, indicando ser um produto de nicho com perfil muito específico. Já o `cartao_platinum` tem boa co-ocorrência com seguros, especialmente `seguro_auto` (1.9%).
- Produtos de investimento têm baixa co-contratação entre si — `CDB + LCI` = 0.1%, `previdência + tesouro` = 0.2%, sugerindo que clientes tendem a escolher apenas um produto de investimento por vez.
- **Oportunidade:** usar padrões de co-contratação como feature de associação no modelo — especialmente o par conta_digital_plus + credito_pessoal como sinalizador de jornada de onboarding.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cold-start: clientes sem histórico
historico_ids = contratos['id_cliente'].unique()
cold_mask = ~clientes['id_cliente'].isin(historico_ids)
cold_data = pd.Series({
    'Com histórico': (~cold_mask).sum(),
    'Cold-Start\n(sem contrato)': cold_mask.sum()
})
axes[0].pie(cold_data.values, labels=cold_data.index,
            autopct='%1.1f%%', colors=['#42A5F5','#EF5350'], startangle=90)
axes[0].set_title('Clientes com/sem Histórico\n(Cold-Start)')

# Produtos com poucos contratos (item cold-start)
prod_cont_count = (ativos.groupby('produto').size()
                   .reindex(all_prod_names, fill_value=0)
                   .sort_values())
COLD_PROD_THRESHOLD = 500
prod_colors = ['#EF5350' if v < COLD_PROD_THRESHOLD else '#42A5F5'
               for v in prod_cont_count.values]
axes[1].barh(prod_cont_count.index, prod_cont_count.values, color=prod_colors)
axes[1].axvline(COLD_PROD_THRESHOLD, color='red', linestyle='--',
                label=f'Limiar ({COLD_PROD_THRESHOLD})')
axes[1].set_title('Contratos Ativos por Produto')
axes[1].set_xlabel('Qtd. Contratos Ativos')
axes[1].legend(fontsize=9)

# Tendência temporal
temporal = (interacoes.groupby('safra')
            .agg(total=('id_interacao','count'),
                 contratos_n=('contratou','sum'),
                 cliques=('clicou','sum'))
            .reset_index())
temporal['cvr'] = temporal['contratos_n'] / temporal['total']
temporal['idx'] = range(len(temporal))
split_idx = temporal[temporal['safra'] < TRAIN_CUTOFF].shape[0] - 0.5

ax2 = axes[2].twinx()
axes[2].bar(temporal['idx'], temporal['contratos_n'],
            color=['#42A5F5' if s < TRAIN_CUTOFF else '#FFA726'
                   for s in temporal['safra']],
            alpha=0.8, label='Contratos')
ax2.plot(temporal['idx'], temporal['cvr']*100, 'o-',
         color='#C62828', linewidth=2, markersize=5, label='CVR (%)')
axes[2].axvline(split_idx, color='purple', linestyle='--', linewidth=2)
axes[2].text(split_idx + 0.2, temporal['contratos_n'].max() * 0.85,
             'TREINO | TESTE', color='purple', fontsize=9, fontweight='bold')
axes[2].set_xticks(temporal['idx'][::3])
axes[2].set_xticklabels(temporal['safra'].iloc[::3].astype(str), rotation=45, fontsize=7)
axes[2].set_title('Tendência Temporal de Contratos')
axes[2].set_ylabel('Contratos')
ax2.set_ylabel('CVR (%)')
axes[2].legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)

plt.suptitle('Cold-Start e Análise Temporal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_coldstart_temporal.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'Clientes cold-start (sem nenhum contrato): {cold_mask.sum():,} ({cold_mask.mean():.1%})')
print(f'Produtos com menos de {COLD_PROD_THRESHOLD} contratos ativos: '
      f'{(prod_cont_count < COLD_PROD_THRESHOLD).sum()}')


**Insight — Cold-Start e Tendência:**
- **~27% dos clientes (13.633) não têm nenhum contrato ativo** — estratégia de cold-start é obrigatória. Esses clientes receberão recomendações baseadas em popularidade segmentada: os produtos mais contratados por clientes do mesmo segmento servem como fallback, aproveitando o que sabemos sobre o perfil mesmo sem histórico individual.
- Apenas 3 produtos têm menos de 500 contratos ativos — o **problema de cold-start de item é baixo** nesta base. O foco da estratégia de cold-start deve ser nos clientes sem histórico, não nos produtos.
- O volume de contratos oscila de forma relativamente estável ao longo das safras, com um **pico de CVR em torno de 202501** que merece atenção — pode ser reflexo de uma campanha pontual ou ruído nos dados sintéticos. Em produção real, esse período seria investigado antes de incluir no treino.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

missing = clientes.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = missing / len(clientes) * 100

axes[0].barh(missing.index, missing_pct.values, color='#EF5350')
for i, (col, pct) in enumerate(zip(missing.index, missing_pct.values)):
    axes[0].text(pct + 0.05, i, f'{pct:.1f}% ({missing[col]:,})', va='center', fontsize=9)
axes[0].set_title('Missing Values em clientes.csv')
axes[0].set_xlabel('% de Registros com Missing')
axes[0].set_xlim(0, 6)

num_cols = ['idade','score_credito','renda_mensal','saldo_medio_conta',
            'qtd_meses_cliente','qtd_produtos_ativos','qtd_transacoes_pix_6m',
            'vlr_total_investimentos','vlr_medio_gasto_cartao',
            'vlr_limite_credito','pct_utilizacao_limite']
corr = clientes[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], linewidths=0.3, cbar_kws={'shrink':0.8})
axes[1].set_title('Correlação entre Features Numéricas')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle('Qualidade dos Dados e Correlações', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_qualidade.png', bbox_inches='tight', dpi=120)
plt.show()

corr_flat = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().reset_index())
corr_flat.columns = ['feat_a','feat_b','corr']
high_corr = corr_flat[corr_flat['corr'].abs() > 0.5].sort_values('corr', ascending=False)
print('Pares com correlação alta (|r| > 0.5):')
print(high_corr.to_string(index=False))


**Insight — Missing e Correlações:**
- Missing values (~3% em 7 colunas) apresentam distribuição uniforme entre as colunas (2.9% a 3.1%), padrão consistente com MCAR (Missing Completely At Random) — imputação pela mediana do segmento é adequada e preserva as distribuições por perfil de cliente.
- Há 8 pares de features com correlação alta (|r| > 0.5). O par mais correlacionado é `renda_mensal + qtd_transacoes_pix_6m` (r=0.93), seguido de `renda_mensal + vlr_medio_gasto_cartao` (r=0.82) e `saldo_medio_conta + qtd_transacoes_pix_6m` (r=0.70). Isso indica que renda é o fator dominante que explica comportamento transacional e de gastos — faz sentido de negócio. O LightGBM lida naturalmente com redundância via feature importance, portanto não removeremos essas features, pois cada uma captura nuances distintas do perfil financeiro.
- `pct_utilizacao_limite` é a única feature com correlações negativas com as demais — clientes com maior renda e saldo tendem a utilizar proporcionalmente menos o limite, o que é esperado e pode ser um sinal relevante de risco de crédito.
- `vlr_total_investimentos` provavelmente tem distribuição zero-inflacionada — muitos clientes básicos sem investimentos ativos. A transformação log1p e uma flag binária ind_investidor capturam melhor esse sinal no modelo.

In [ ]:
print('=== Missing Values — Demais Bases ===\n')
for nome, df in [('interacoes', interacoes), ('produtos', produtos), ('contratos', contratos)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f'{nome}.csv:')
    print(missing if len(missing) > 0 else '  Sem missing values')
    print()

**Insight — Missing Values nas demais bases:**
- `interacoes.csv` e `produtos.csv` não apresentam missing values — dados completos para modelagem direta.
- `contratos_ativos.csv` também não tem missing — o campo `status` está sempre preenchido, o que é essencial para a regra de exclusão de produtos já ativos no ranking.
- A estratégia de imputação por mediana do segmento se aplica **exclusivamente** a `clientes.csv`, onde os campos socioeconômicos têm ~3% de ausência.

## 3. Feature Engineering

### Split Temporal
- **Treino:** safras 202401–202509 (21 meses, ~87.5% da linha do tempo)
- **Teste:** safras 202510–202512 (3 meses, ~12.5%)
- **Justificativa do corte:** 3 safras de teste geram ~60k interações e ~4k contratos únicos — volume suficiente para estimar Precision@5 e NDCG@5 com baixa variância. Abrir mais safras de teste reduziria o treino e prejudicaria modelos sequencialmente dependentes.

### Estratégia de Cold-Start
- **Clientes sem interação histórica:** fallback para ranking de popularidade dentro do segmento.
- **Produtos com poucos dados:** feature `log_hist_contracts` naturalmente penaliza produtos raros; CF recebe peso zero para itens não vistos.


In [ ]:
# --- Split temporal ---
train_inter = interacoes[interacoes['safra'] < TRAIN_CUTOFF].copy()
test_inter  = interacoes[interacoes['safra'].isin(TEST_SAFRAS)].copy()
train_contratos = contratos[contratos['data_contratacao'] < TEST_START_DATE].copy()

print(f'Interações treino:  {len(train_inter):,}')
print(f'Interações teste:   {len(test_inter):,}  ({len(test_inter)/len(interacoes):.1%} do total)')
print(f'Contratos treino:   {len(train_contratos):,}')

print(f'\nContratações no conjunto de treino: {train_inter.contratou.sum():,} '
      f'({train_inter.contratou.mean():.2%} das interações)')
print(f'Contratações no conjunto de teste:  {test_inter.contratou.sum():,} '
      f'({test_inter.contratou.mean():.2%} das interações)')

print(f'\nClientes distintos no treino: {train_inter.id_cliente.nunique():,}')
print(f'Clientes distintos no teste:  {test_inter.id_cliente.nunique():,}')
print(f'Clientes com contrato no teste: '
      f'{test_inter[test_inter.contratou==1].id_cliente.nunique():,} '
      f'({test_inter[test_inter.contratou==1].id_cliente.nunique()/test_inter.id_cliente.nunique():.1%} dos clientes de teste)')


### Features do Cliente

As features do cliente cobrem 4 dimensões do perfil financeiro: (1) dados demográficos e de relacionamento com o banco (idade, tempo de cliente, segmento, UF); (2) capacidade financeira (renda, saldo, score de crédito, limite); (3) comportamento transacional (gastos por categoria, PIX, utilização do limite, débito automático, inatividade); (4) perfil de investidor (volume total e indicador binário). Gênero foi excluído por ser dado sensível conforme Art. 11 da LGPD.

In [ ]:
# --- Features do cliente ---
client_feats = clientes.copy()

# Imputação pela mediana do segmento
num_miss = ['score_credito','renda_mensal','vlr_total_investimentos',
            'vlr_medio_gasto_saude','vlr_medio_gasto_educacao',
            'vlr_medio_gasto_lazer','qtd_dias_inatividade']
for col in num_miss:
    med = client_feats.groupby('segmento')[col].transform('median')
    client_feats[col] = client_feats[col].fillna(med)

# Indicador binário para vlr_total_investimentos (distribuição zero-inflacionada)
client_feats['ind_investidor'] = (client_feats['vlr_total_investimentos'] > 0).astype(int)

# Encoding ordinal para segmento
SEG_ORDER = {'basico': 0, 'intermediario': 1, 'premium': 2}
client_feats['segmento_enc'] = client_feats['segmento'].map(SEG_ORDER)

# Variáveis categóricas
# Nota: gênero não é utilizado como feature — dado sensível conforme Art. 11 da LGPD.
# Em produção, essa decisão deve ser validada pelo time jurídico e de compliance.
le_uf         = LabelEncoder()
le_canal_pref = LabelEncoder()
client_feats['uf_enc']         = le_uf.fit_transform(client_feats['uf'])
client_feats['canal_pref_enc'] = le_canal_pref.fit_transform(client_feats['canal_preferencial'])

# Feature derivada
client_feats['pct_gasto_cartao'] = (client_feats['vlr_medio_gasto_cartao'] /
                                     client_feats['renda_mensal'].clip(lower=1))
    # proxy de intensidade de uso do cartão

CLIENT_FEATURE_COLS = [
    'idade','uf_enc','segmento_enc','score_credito',
    'renda_mensal','saldo_medio_conta','qtd_meses_cliente',
    'qtd_produtos_ativos','qtd_transacoes_pix_6m',
    'vlr_total_investimentos','ind_investidor',
    'vlr_medio_gasto_cartao','vlr_medio_gasto_alimentacao',
    'vlr_medio_gasto_transporte','vlr_medio_gasto_saude',
    'vlr_medio_gasto_educacao','vlr_medio_gasto_lazer',
    'ind_debito_automatico','qtd_dias_inatividade',
    'vlr_limite_credito','pct_utilizacao_limite',
    'canal_pref_enc','pct_gasto_cartao'
]

print(f'Features do cliente: {len(CLIENT_FEATURE_COLS)}')
print(client_feats[CLIENT_FEATURE_COLS].describe().T[['mean','std','min','max']].round(3))

### Features do Produto

As features do produto combinam metadados financeiros estáticos (receita_media, custo_aquisicao, margem, categoria) com performance histórica calculada sobre o conjunto de treino (hist_ctr, hist_cvr, hist_contracts, hist_revenue_mean). Calcular a performance histórica apenas sobre o treino evita vazamento de dados do período de teste. A variável publico_alvo é textual e exigiria processamento de linguagem natural (TF-IDF ou embeddings) para ser utilizada — identificada como melhoria futura de médio prazo.

In [ ]:
# --- Features do produto (metadados + performance histórica de treino) ---
product_feats = produtos.copy()

prod_hist = (train_inter.groupby('produto')
             .agg(hist_views=('id_interacao','count'),
                  hist_clicks=('clicou','sum'),
                  hist_contracts=('contratou','sum'),
                  hist_revenue=('receita_gerada','sum'))
             .reset_index())

prod_hist['hist_ctr'] = prod_hist['hist_clicks'] / prod_hist['hist_views']
prod_hist['hist_cvr'] = prod_hist['hist_contracts'] / prod_hist['hist_views']

# Receita média por contrato — divisão direta quando há contratos, zero caso contrário
prod_hist['hist_revenue_mean'] = np.where(
    prod_hist['hist_contracts'] > 0,
    prod_hist['hist_revenue'] / prod_hist['hist_contracts'],
    0
)

product_feats = product_feats.merge(prod_hist, on='produto', how='left')

le_cat = LabelEncoder()
product_feats['categoria_enc'] = le_cat.fit_transform(product_feats['categoria'])

PRODUCT_FEATURE_COLS = [
    'categoria_enc','receita_media','custo_aquisicao','margem',
    'hist_ctr','hist_cvr','hist_contracts','hist_revenue_mean'
]

print(f'Features do produto: {len(PRODUCT_FEATURE_COLS)}')
print(product_feats[['produto'] + PRODUCT_FEATURE_COLS].to_string(index=False))

### Features de Interação — Acumulado Histórico com Janela Temporal

As features de interação são calculadas usando exclusivamente dados anteriores à safra de cada linha — cada registro (cliente, produto, safra) só enxerga o histórico acumulado até a safra anterior. Essa abordagem respeita a ordem causal e evita leakage temporal: o modelo nunca usa informação futura para aprender.
Linhas sem histórico anterior (primeira aparição do par cliente-produto) recebem zero, representando o cenário de cold-start de interação.

Pontos de atenção para produção:
- O acumulado histórico completo trata igualmente interações antigas e recentes — um cliente que clicou 10 vezes em 2024 mas parou em 2025 parece igual a um que clicou 10 vezes recentemente. Em produção, janelas combinadas (último mês, últimos 3 meses, acumulado total) capturam melhor a evolução do comportamento.
- Uma janela rolante com decay temporal exponencial daria mais peso ao comportamento recente, aumentando a sensibilidade do modelo a mudanças de interesse do cliente.
- Ambas as melhorias estão identificadas no roadmap de evolução do sistema.

In [ ]:
# --- Correção de viés de posição (IPW) ---
# Calculado apenas sobre dados de treino
pos_perf = (train_inter.groupby('posicao_exibicao')
            .agg(views=('id_interacao','count'), clicks=('clicou','sum'))
            .reset_index())
pos_perf['ctr_pos']   = pos_perf['clicks'] / pos_perf['views']
ctr_pos1 = pos_perf.loc[pos_perf['posicao_exibicao']==1, 'ctr_pos'].values[0]
pos_perf['propensity'] = pos_perf['ctr_pos'] / ctr_pos1
pos_perf['ipw']        = 1.0 / pos_perf['propensity'].clip(lower=0.01)
pos_perf['ipw_norm']   = pos_perf['ipw'] / pos_perf['ipw'].mean()
    # IPW normalizado — interações em posições piores recebem peso maior no treino,
    # compensando o viés de exposição e permitindo que o modelo aprenda preferência real

position_propensity = pos_perf[['posicao_exibicao','propensity','ipw_norm']].copy()

# --- Features de interação com janela temporal (sem leakage) ---
# Abordagem vetorizada: cumsum() - valor_atual = acumulado até linha anterior
# Equivalente a shift(1).expanding().sum() mas O(n) em vez de O(n × n_grupos)

train_inter_sorted = train_inter.sort_values(['id_cliente','produto','safra']).copy()
grp = train_inter_sorted.groupby(['id_cliente','produto'])

# cumcount() = nº de linhas anteriores no grupo = n_views antes da linha atual
train_inter_sorted['n_views_hist']     = grp.cumcount()
train_inter_sorted['n_clicks_hist']    = (grp['clicou'].cumsum()
                                            - train_inter_sorted['clicou'])
train_inter_sorted['n_contracts_hist'] = (grp['contratou'].cumsum()
                                            - train_inter_sorted['contratou'])

# avg_position_hist: média das posições anteriores (acumulado / contagem)
_cum_pos = (grp['posicao_exibicao'].cumsum()
            - train_inter_sorted['posicao_exibicao'])
train_inter_sorted['avg_position_hist'] = np.where(
    train_inter_sorted['n_views_hist'] > 0,
    _cum_pos / train_inter_sorted['n_views_hist'],
    0
)

# Taxas históricas — evita divisão por zero
train_inter_sorted['click_rate_hist'] = np.where(
    train_inter_sorted['n_views_hist'] > 0,
    train_inter_sorted['n_clicks_hist'] / train_inter_sorted['n_views_hist'],
    0
)
train_inter_sorted['contract_rate_hist'] = np.where(
    train_inter_sorted['n_views_hist'] > 0,
    train_inter_sorted['n_contracts_hist'] / train_inter_sorted['n_views_hist'],
    0
)

# Features agregadas por cliente com janela temporal (também vetorizado)
client_safra_agg = (train_inter_sorted.groupby(['id_cliente','safra'])
    .agg(client_views=('id_interacao','count'),
         client_clicks=('clicou','sum'),
         client_contracts=('contratou','sum'))
    .reset_index()
    .sort_values(['id_cliente','safra']))

grp_c = client_safra_agg.groupby('id_cliente')
client_safra_agg['client_total_views_hist']     = (grp_c['client_views'].cumsum()
                                                    - client_safra_agg['client_views'])
client_safra_agg['client_total_clicks_hist']    = (grp_c['client_clicks'].cumsum()
                                                    - client_safra_agg['client_clicks'])
client_safra_agg['client_total_contracts_hist'] = (grp_c['client_contracts'].cumsum()
                                                    - client_safra_agg['client_contracts'])

client_safra_agg['client_ctr_hist'] = np.where(
    client_safra_agg['client_total_views_hist'] > 0,
    client_safra_agg['client_total_clicks_hist'] / client_safra_agg['client_total_views_hist'],
    0
)
client_safra_agg['client_cvr_hist'] = np.where(
    client_safra_agg['client_total_views_hist'] > 0,
    client_safra_agg['client_total_contracts_hist'] / client_safra_agg['client_total_views_hist'],
    0
)

# Merge das features no dataset de treino
train_inter_feat = train_inter_sorted.merge(
    client_safra_agg[['id_cliente','safra',
                       'client_total_views_hist','client_total_clicks_hist',
                       'client_ctr_hist','client_cvr_hist']],
    on=['id_cliente','safra'], how='left')

INTER_FEATURE_COLS = [
    'n_views_hist','n_clicks_hist','n_contracts_hist',
    'avg_position_hist','click_rate_hist','contract_rate_hist'
]
CLIENT_AGG_COLS = [
    'client_total_views_hist','client_total_clicks_hist',
    'client_ctr_hist','client_cvr_hist'
]

print(f'Features de interação (par cliente-produto-safra): {len(INTER_FEATURE_COLS)}')
print(f'Features agregadas do cliente: {len(CLIENT_AGG_COLS)}')
print(f'Dataset de treino com features temporais: {train_inter_feat.shape}')
print(f'% de linhas com histórico zerado (primeira aparição): '
      f'{(train_inter_feat["n_views_hist"]==0).mean():.1%}')

In [ ]:
# --- Montagem do dataset de treino para LightGBM ---
# Cada linha = uma interação do período de treino (com features temporais sem leakage)
lgb_train = train_inter_feat[['id_cliente','produto','posicao_exibicao',
                               'canal','contratou','safra']
                              + INTER_FEATURE_COLS + CLIENT_AGG_COLS].copy()

lgb_train = lgb_train.merge(
    client_feats[['id_cliente'] + CLIENT_FEATURE_COLS], on='id_cliente', how='left')
lgb_train = lgb_train.merge(
    product_feats[['produto'] + PRODUCT_FEATURE_COLS], on='produto', how='left')
lgb_train = lgb_train.merge(position_propensity, on='posicao_exibicao', how='left')
lgb_train['propensity'] = lgb_train['propensity'].fillna(pos_perf['propensity'].min())
lgb_train['ipw_norm']   = lgb_train['ipw_norm'].fillna(pos_perf['ipw_norm'].max())

le_canal = LabelEncoder()
lgb_train['canal_enc'] = le_canal.fit_transform(lgb_train['canal'])

FEATURE_COLS = (CLIENT_FEATURE_COLS + PRODUCT_FEATURE_COLS +
                INTER_FEATURE_COLS + CLIENT_AGG_COLS +
                ['canal_enc'])

X_all = lgb_train[FEATURE_COLS]
y_all = lgb_train['contratou']
w_all = lgb_train['ipw_norm']

# Split interno treino/validação por safra para early stopping
VAL_SAFRAS_INNER = [202507, 202508, 202509]
tr_mask  = ~lgb_train['safra'].isin(VAL_SAFRAS_INNER)
val_mask =  lgb_train['safra'].isin(VAL_SAFRAS_INNER)

X_tr, y_tr, w_tr = X_all[tr_mask], y_all[tr_mask], w_all[tr_mask]
X_val, y_val     = X_all[val_mask], y_all[val_mask]

# --- Agregados para inferência (full training history — usados na geração final) ---
# Para prever para todos os 50k clientes, usamos o acumulado completo do treino,
# que é o valor correto a ser utilizado em produção (dados até o momento da predição)
agg_inter = (train_inter.groupby(['id_cliente','produto'])
             .agg(n_views_hist=('id_interacao','count'),
                  n_clicks_hist=('clicou','sum'),
                  n_contracts_hist=('contratou','sum'),
                  avg_position_hist=('posicao_exibicao','mean'))
             .reset_index())
agg_inter['click_rate_hist']    = np.where(agg_inter['n_views_hist'] > 0,
    agg_inter['n_clicks_hist']    / agg_inter['n_views_hist'], 0)
agg_inter['contract_rate_hist'] = np.where(agg_inter['n_views_hist'] > 0,
    agg_inter['n_contracts_hist'] / agg_inter['n_views_hist'], 0)

_c = (train_inter.groupby('id_cliente')
      .agg(client_total_views_hist=('id_interacao','count'),
           client_total_clicks_hist=('clicou','sum'),
           _contracts=('contratou','sum'))
      .reset_index())
_c['client_ctr_hist'] = _c['client_total_clicks_hist'] / _c['client_total_views_hist'].clip(lower=1)
_c['client_cvr_hist'] = _c['_contracts'] / _c['client_total_views_hist'].clip(lower=1)
client_agg = _c.drop(columns='_contracts')

print(f'Dataset total de treino:      {X_all.shape}')
print(f'  → Sub-treino (inner):       {X_tr.shape}')
print(f'  → Validação early stopping: {X_val.shape}')
print(f'Taxa de contratação treino:   {y_all.mean():.4f} ({y_all.sum():,} contratos)')
print(f'Feature cols totais:          {len(FEATURE_COLS)}')

## 4. Modelagem

### Abordagem
Implementamos **3 baselines** e **2 modelos principais**:

| Modelo | Tipo | Descrição |
|--------|------|-----------|
| Baseline Popularidade | Heurística | Ordena produtos por total de contratos no treino |
| Baseline Aleatório | Heurística | Ordem aleatória (lower bound) |
| Baseline Segmento | Heurística | Ordena por CVR dentro do segmento do cliente |
| **LightGBM** | Classificação supervisionada | Prediz P(contratar) para cada par (cliente, produto) usando todas as features + IPW |
| **CF-SVD** | Filtragem colaborativa | Fatoração matricial (TruncatedSVD) sobre sinais implícitos de interação |


In [ ]:
# --- Baselines ---
all_produtos_list = produtos['produto'].tolist()

# Baseline 1: Popularidade global
pop_ranking = (train_inter[train_inter['contratou']==1]
               .groupby('produto').size()
               .reindex(all_produtos_list, fill_value=0)
               .sort_values(ascending=False)
               .index.tolist())

# Baseline 2: Aleatório (seed por cliente para reprodutibilidade)
def get_random_recs(cliente_id, excl=None, n=20):
    avail = [p for p in all_produtos_list if p not in (excl or set())]
    rng = random.Random(SEED + hash(str(cliente_id)) % 100000)
    rng.shuffle(avail)
    return avail[:n]

# Baseline 3: Popularidade por segmento
seg_cvr = (train_inter.merge(clientes[['id_cliente','segmento']], on='id_cliente')
           .groupby(['segmento','produto'])['contratou'].mean()
           .reset_index())
seg_rankings = {}
for seg in clientes['segmento'].unique():
    seg_df = seg_cvr[seg_cvr['segmento']==seg].sort_values('contratou', ascending=False)
    seg_rankings[seg] = seg_df['produto'].tolist()

def get_segment_recs(cliente_id, excl=None, n=20):
    seg_vals = clientes.loc[clientes['id_cliente']==cliente_id, 'segmento']
    if len(seg_vals) == 0:
        base = pop_ranking
    else:
        base = seg_rankings.get(seg_vals.values[0], pop_ranking)
    return [p for p in base if p not in (excl or set())][:n]

print('Baselines configurados.')
print(f'Top-5 popularidade global: {pop_ranking[:5]}')
for seg, rank in seg_rankings.items():
    print(f'Top-3 segmento {seg}: {rank[:3]}')


### 4.1 Abordagem 1 — LightGBM (Classificação Binária com IPW)

**Justificativa técnica:** LightGBM é altamente eficiente em dados tabulares heterogêneos, captura interações não-lineares entre features e permite incorporar o peso de amostra (IPW) diretamente no treinamento. A formulação como classificação binária (contratou=1/0) é natural para o problema e produz scores calibrados de probabilidade.

**Pipeline:**
1. Cada linha do dataset = 1 interação (par cliente × produto × safra) no período de treino
2. Features: perfil do cliente + metadados do produto + histórico de interação com janela temporal + canal de interação
3. Peso de amostra = IPW normalizado (corrige viés de posição)
4. Early stopping em validação temporal (safras 202507-202509)
5. Inferência: criar todos os pares (cliente, produto elegível) e ordenar por score

**Limitações e decisões de design:**

- **Modelo point-wise** — não captura diretamente a relação entre itens no ranking. Modelos list-wise como LambdaMART seriam mais sofisticados para ranking direto, mas com maior complexidade de implementação.
- **Modelo único para todos os produtos** — uma abordagem mais sofisticada segmentaria os produtos em 3 grupos: (1) produtos populares com volume suficiente para modelos individuais (conta_digital_plus, credito_pessoal, cartao_platinum); (2) produtos de nicho agrupados por categoria (cartao_black, consorcio_imovel, uso_lastro_limite); (3) produtos em cold-start com fallback de popularidade segmentada. A escolha do modelo único é justificada pelo tempo disponível e pelo risco de underfitting em produtos raros como cartao_black (~37 contratos ativos), mas em produção essa segmentação seria o próximo passo natural.
- **Features de interação histórica** (n_clicks, n_contracts) podem introduzir viés de popularidade mesmo com IPW — em produção, features prospectivas baseadas em perfil financeiro recente devem complementar o sinal histórico.


### Decisões de Design do LightGBM

- **posicao_exibicao e propensity foram removidas de FEATURE_COLS** — a posição não é conhecida no momento da inferência e incluí-la criaria um modelo que aprende a recomendar produtos que já aparecem em posições boas, perpetuando o viés que o IPW tenta corrigir. O IPW é usado exclusivamente como peso de amostra durante o treino.
- **scale_pos_weight corrige o desbalanceamento de classes** — com taxa de contratação de ~0.5%, sem esse ajuste o modelo tenderia a prever sempre zero e ainda assim ter alta acurácia, mas sem aprender os padrões reais de contratação.
- **canal_enc permanece como feature** — o canal de interação (app_busca, app_home, etc.) é conhecido no momento da inferência e captura o contexto de engajamento do cliente.

In [ ]:
# --- Busca de hiperparâmetros ---
# Calcular desbalanceamento e criar datasets LightGBM
n_neg = (y_tr == 0).sum()
n_pos = (y_tr == 1).sum()
scale_pw = n_neg / n_pos

train_ds = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, free_raw_data=False)
val_ds   = lgb.Dataset(X_val, label=y_val, reference=train_ds, free_raw_data=False)

lgb_params_base = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'scale_pos_weight': scale_pw,
    'feature_pre_filter': False,
    'verbose': -1,
    'seed': SEED,
    'n_jobs': -1,
}

param_candidates = [
    {'num_leaves': 31,  'learning_rate': 0.10, 'min_child_samples': 50},
    {'num_leaves': 63,  'learning_rate': 0.05, 'min_child_samples': 50},
    {'num_leaves': 127, 'learning_rate': 0.05, 'min_child_samples': 30},
]

best_auc_hp, best_params_hp, best_model_hp = 0.0, {}, None
for params in param_candidates:
    p = {**lgb_params_base, **params}
    m = lgb.train(p, train_ds, num_boost_round=400,
                  valid_sets=[val_ds],
                  callbacks=[lgb.early_stopping(30, verbose=False),
                              lgb.log_evaluation(9999)])
    pred = m.predict(X_val)
    auc  = roc_auc_score(y_val, pred)
    print(f'  leaves={params["num_leaves"]:3d} | lr={params["learning_rate"]} | '
          f'min_child={params["min_child_samples"]:2d} → AUC={auc:.4f}')
    if auc > best_auc_hp:
        best_auc_hp, best_params_hp, best_model_hp = auc, params, m

print(f'\nMelhor config: {best_params_hp} → AUC={best_auc_hp:.4f}')

**Busca de Hiperparâmetros — Grid Manual Reduzido**

A busca cobre 3 candidatos que representam os extremos do espaço de complexidade do modelo — simples, médio e complexo — permitindo identificar o ponto de equilíbrio entre underfitting e overfitting com custo computacional baixo.

**Resultado:**
- `num_leaves=31, lr=0.10` → AUC=0.7548 — modelo muito simples, underfitting: árvores rasas com learning rate alto convergem rápido mas não capturam a complexidade do problema.
- `num_leaves=63, lr=0.05` → AUC=0.8155 — melhor configuração: equilíbrio entre complexidade e generalização.
- `num_leaves=127, lr=0.05` → AUC=0.8043 — modelo mais complexo com leve overfitting: árvores mais profundas memorizam padrões do treino que não generalizam para o período de validação.

**Decisão consciente de escopo:** a busca foi intencionalmente limitada a 3 candidatos dado o tempo disponível. Em produção, usaríamos Optuna com otimização bayesiana — que aprende dos resultados anteriores para guiar a busca de forma inteligente — combinado com validação cruzada temporal para garantir robustez estatística. Isso permitiria explorar um espaço muito maior de hiperparâmetros (learning rate, regularização, profundidade, subsampling) com menor custo computacional que um grid search completo.

In [ ]:
# --- Treinamento final com melhor configuração ---
n_neg = (y_tr == 0).sum()
n_pos = (y_tr == 1).sum()
scale_pw = n_neg / n_pos
print(f'Desbalanceamento: {n_neg:,} negativos / {n_pos:,} positivos (ratio: {scale_pw:.1f}x)')

final_params = {**lgb_params_base, **best_params_hp}

train_ds = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, free_raw_data=False)
val_ds   = lgb.Dataset(X_val, label=y_val, reference=train_ds, free_raw_data=False)

lgb_model = lgb.train(
    final_params,
    train_ds,
    num_boost_round=600,
    valid_sets=[train_ds, val_ds],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)

val_pred = lgb_model.predict(X_val)
val_auc  = roc_auc_score(y_val, val_pred)
print(f'LightGBM — AUC validação (inner): {val_auc:.4f}')
print(f'Best iteration: {lgb_model.best_iteration}')

**Resultado do treinamento final:**
- AUC treino: 0.9486 vs AUC validação: 0.8155 — resultado sólido para um problema de recomendação bancária com dados altamente desbalanceados (107.7x). AUC acima de 0.80 indica boa capacidade de separar clientes que vão contratar dos que não vão.
- A diferença entre treino e validação (~0.13) é esperada em dados tabulares com features históricas e não configura overfitting preocupante — o early stopping parou em 148 iterações (de 600 possíveis), indicando que o modelo convergiu antes de memorizar o treino.
- O `scale_pos_weight=107.7` foi essencial para que o modelo aprendesse os padrões reais de contratação — sem ele, prever sempre zero já daria ~99% de acurácia mas sem nenhum valor preditivo.

In [ ]:
# --- Feature importance ---
feat_imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 9))
top_n = 25
feat_top = feat_imp.head(top_n)
bars = ax.barh(feat_top['feature'][::-1], feat_top['importance'][::-1],
               color=sns.color_palette('Blues_r', top_n))
ax.set_title(f'Top {top_n} Features por Importância (Gain) — LightGBM', fontweight='bold')
ax.set_xlabel('Importância (Gain acumulado)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/lgbm_feature_importance.png', bbox_inches='tight', dpi=120)
plt.show()

print('Top 15 features:')
print(feat_imp.head(15).to_string(index=False))


**Interpretação da Feature Importance — Visão de Negócio:**
- **hist_ctr domina com folga** — a taxa histórica de clique do par (cliente, produto) é de longe o sinal mais preditivo. Clientes que clicaram em um produto no passado têm alta propensão a contratar. Ponto de atenção: essa dependência torna o modelo mais fraco para clientes com pouco histórico de interação (cold-start), reforçando a importância da estratégia de fallback por segmento.
- **canal_enc e segmento_enc em 2º e 3º** — o contexto de onde o cliente está (canal) e quem ele é (segmento) são os sinais de perfil mais determinantes para a recomendação, confirmando os padrões identificados na EDA.
- **vlr_medio_gasto_educacao em 4º acima de renda_mensal** — resultado inesperado que merece investigação em produção. Gasto com educação pode estar funcionando como proxy de perfil socioeconômico ou capturando um padrão específico dos dados sintéticos. Em produção real, aplicaríamos testes de estabilidade para verificar se esse sinal se mantém ao longo do tempo.
- **avg_position_hist em 6º** — a posição média histórica em que o produto foi exibido ainda influencia o modelo, indicando que o IPW não eliminou completamente o viés de posição — apenas o atenuou. Em produção, técnicas mais sofisticadas de debiasing como DLA (Deep Learning Approach) ou IPS com propensity scores estimados por modelo dedicado poderiam reduzir esse efeito.
- **Features com importância próxima de zero** (pct_gasto_cartao, vlr_medio_gasto_alimentacao, vlr_medio_gasto_transporte, client_ctr_hist) — contribuem pouco para o modelo e são candidatas a remoção em uma próxima iteração para simplificar o pipeline e reduzir ruído.

### 4.2 Abordagem 2 — Filtragem Colaborativa com Fatoração Matricial (SVD)

**Justificativa técnica:** A CF captura padrões latentes de co-interesse entre clientes sem depender de features explícitas — complementando o LightGBM que é forte em perfil mas menos sensível a padrões coletivos de comportamento. Clientes com histórico de interação similar recebem recomendações similares. A análise de co-contratação da EDA (ex: conta_digital_plus + credito_pessoal = 4.3% da base) motiva essa abordagem — o SVD aprende esses padrões automaticamente sem precisar explicitar as regras. Usamos TruncatedSVD sobre uma matriz de feedback implícito ponderado, computacionalmente eficiente para dados esparsos.

**Decisões de design:**
- **Pesos do sinal implícito:** contrato=5, clique=1 — contratação é um sinal de interesse muito mais forte e definitivo do que um clique, justificando peso 5× maior.
- **k=15 componentes latentes:** escolhido empiricamente como equilíbrio entre capacidade de representação e risco de overfitting. Em produção, validaríamos k no intervalo [10, 50] usando NDCG@5 como critério.

**Pipeline:**
1. Construir matriz esparsa cliente × produto com sinal implícito agregado
2. Fatorar com TruncatedSVD (k=15 componentes latentes)
3. Score para (cliente, produto) = produto interno dos fatores latentes
4. Cold-start: clientes sem histórico recebem ranqueamento por popularidade segmentada

**Limitações:** não usa features explícitas de perfil — cold-start severo para clientes e produtos novos. Não modela viés de posição diretamente. Sensível à esparsidade da matriz — clientes com poucas interações têm representações latentes menos precisas.

In [ ]:
# --- Matriz de feedback implícito ---
# Sinal: contratou=1 → peso 5, clicou=1 mas não contratou → peso 1
client_ids_all = clientes['id_cliente'].tolist()
client_to_idx  = {c: i for i, c in enumerate(client_ids_all)}
prod_to_idx    = {p: i for i, p in enumerate(all_produtos_list)}
idx_to_prod    = {i: p for p, i in prod_to_idx.items()}
n_clients = len(client_ids_all)
n_prods   = len(all_produtos_list)

# Pesos por interação
train_inter['cf_weight'] = (train_inter['contratou'] * 5 +
                             train_inter['clicou'] * (1 - train_inter['contratou']) * 1)
cf_agg = (train_inter[train_inter['cf_weight'] > 0]
          .groupby(['id_cliente','produto'])['cf_weight'].sum()
          .reset_index())
cf_agg = cf_agg[cf_agg['id_cliente'].isin(client_to_idx) &
                cf_agg['produto'].isin(prod_to_idx)]

rows = [client_to_idx[c] for c in cf_agg['id_cliente']]
cols = [prod_to_idx[p]   for p in cf_agg['produto']]
vals = cf_agg['cf_weight'].values

user_item_matrix = csr_matrix((vals, (rows, cols)), shape=(n_clients, n_prods))
density = user_item_matrix.nnz / (n_clients * n_prods)
print(f'Matriz usuário-item: {user_item_matrix.shape}')
print(f'Densidade: {density:.4f} ({user_item_matrix.nnz:,} entradas não-zero)')


In [ ]:
# --- TruncatedSVD ---
N_COMPONENTS = min(15, n_prods - 1)   # SVD requer k < n_features (20 produtos)
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED, n_iter=10)
user_factors = svd.fit_transform(user_item_matrix)  # (n_clients, 15)
item_factors = svd.components_.T                     # (n_prods, 15)

explained_var = svd.explained_variance_ratio_.sum()
print(f'Variância explicada com {N_COMPONENTS} componentes: {explained_var:.2%}')

# Scores: (n_clients, n_prods)
cf_scores_matrix = user_factors @ item_factors.T

# Componentes explicados por dimensão
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, N_COMPONENTS+1), svd.explained_variance_ratio_ * 100)
ax.plot(range(1, N_COMPONENTS+1),
        np.cumsum(svd.explained_variance_ratio_) * 100,
        'o-', color='red', markersize=3, label='Variância acumulada')
ax.set_xlabel('Componente SVD')
ax.set_ylabel('Variância Explicada (%)')
ax.set_title('Variância Explicada por Componente — CF-SVD')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cf_svd_variance.png', bbox_inches='tight', dpi=120)
plt.show()


**Interpretação da Fatoração Matricial:**
- **Densidade de 3.81%** — a matriz é altamente esparsa: 96.19% dos pares (cliente, produto) não têm nenhum sinal de interação. Isso confirma a importância da estratégia de cold-start por popularidade segmentada para clientes sem histórico.
- **99.45% de variância explicada com 15 componentes** — os fatores latentes capturam quase toda a estrutura de preferências da base. Os 2 primeiros componentes já explicam ~55% da variância, sugerindo que as preferências são dominadas por padrões populacionais fortes (segmento, produtos mais contratados). Em dados reais com mais diversidade comportamental, esperaríamos uma curva mais gradual exigindo mais componentes para capturar nuances individuais.
- **Complementaridade com o LightGBM:** o CF-SVD captura padrões coletivos de co-interesse (quem se comporta como você tende a contratar X), enquanto o LightGBM captura o perfil individual explícito (sua renda e score sugerem X). O modelo híbrido combina os dois sinais.

## 5. Avaliação e Comparação

### Framework de Avaliação
- **Conjunto de teste:** clientes com pelo menos 1 contratação nas safras 202510–202512
- **Ground truth:** produtos contratados no período de teste por cada cliente
- **Exclusão:** produtos com `status='ativo'` contratados antes de 2025-10-01 — não elegíveis para recomendação no momento do teste
- **Métricas primárias:** Precision@5 e NDCG@5 — focadas nas 5 posições visíveis sem scroll, onde a ordem do carrossel realmente impacta a contratação
- **Métricas complementares:** Hit Rate@5 e MRR — escolhidas por serem diretamente interpretáveis pelo negócio

### Interpretação das Métricas
- **Precision@5** — de cada 5 produtos recomendados nas posições visíveis, quantos o cliente realmente contratou. Ex: Precision@5=0.10 significa que 1 em cada 10 recomendações foi um acerto.
- **NDCG@5** — similar ao Precision@5 mas dá mais peso para acertos nas primeiras posições — um acerto na posição 1 vale mais do que na posição 5, refletindo o comportamento real do carrossel.
- **Hit Rate@5** — percentual de clientes para os quais o modelo acertou pelo menos 1 produto nos top 5. Responde à pergunta: para quantos clientes o modelo foi útil?
- **MRR (Mean Reciprocal Rank)** — em qual posição média aparece o primeiro produto acertado. Ex: MRR=0.25 significa que em média o primeiro acerto aparece na posição 4.

In [ ]:
# --- Funções de métricas ---
def precision_at_k(rec, rel, k):
    hits = sum(1 for r in rec[:k] if r in rel)
    return hits / k

def hit_rate_at_k(rec, rel, k):
    return int(any(r in rel for r in rec[:k]))

def ndcg_at_k(rec, rel, k):
    dcg  = sum(1/np.log2(i+2) for i, r in enumerate(rec[:k]) if r in rel)
    idcg = sum(1/np.log2(i+2) for i in range(min(len(rel), k)))
    return dcg/idcg if idcg > 0 else 0.0

def mrr_score(rec, rel):
    for i, r in enumerate(rec):
        if r in rel: return 1/(i+1)
    return 0.0

def evaluate_model(get_recs_fn, gt_dict, excl_dict, k_vals=[5]):
    metrics = {k: {'precision':[], 'hit_rate':[], 'ndcg':[], 'mrr':[]}
               for k in k_vals}
    for cid, rel_set in gt_dict.items():
        excl = excl_dict.get(cid, set())
        rel  = rel_set - excl
        if not rel: continue
        recs = get_recs_fn(cid, excl)
        for k in k_vals:
            metrics[k]['precision'].append(precision_at_k(recs, rel, k))
            metrics[k]['hit_rate'].append(hit_rate_at_k(recs, rel, k))
            metrics[k]['ndcg'].append(ndcg_at_k(recs, rel, k))
            metrics[k]['mrr'].append(mrr_score(recs, rel))
    summary = {k: {m: np.mean(v) for m, v in metrics[k].items()} for k in k_vals}
    return summary

print('Funções de métricas definidas.')

In [ ]:
# --- Preparação do conjunto de teste ---
# Ground truth: contratos realizados no período de teste
test_gt_raw = (test_inter[test_inter['contratou']==1]
               .groupby('id_cliente')['produto']
               .apply(set).to_dict())

# Produtos a excluir: status='ativo' contratados ANTES do teste
active_pre_test = (contratos[(contratos['status']=='ativo') &
                              (contratos['data_contratacao'] < TEST_START_DATE)]
                   .groupby('id_cliente')['produto']
                   .apply(set).to_dict())

# Filtrar clientes que ainda têm ao menos 1 contrato válido após exclusão
test_gt = {}
for cid, rel in test_gt_raw.items():
    excl = active_pre_test.get(cid, set())
    valid_rel = rel - excl
    if valid_rel:
        test_gt[cid] = valid_rel

test_client_ids = list(test_gt.keys())
print(f'Clientes com contratações válidas no teste: {len(test_client_ids):,}')
print(f'Total de contratos de ground truth: {sum(len(v) for v in test_gt.values()):,}')
print(f'Produtos únicos no ground truth: '
      f'{len(set(p for s in test_gt.values() for p in s))}')

# Impacto da regra de exclusão
contratos_excluidos = sum(
    len(test_gt_raw.get(cid, set()) & active_pre_test.get(cid, set()))
    for cid in test_gt_raw
)
print(f'\nContratos excluídos por já estarem ativos antes do teste: '
      f'{contratos_excluidos:,}')
print(f'Clientes removidos por não terem contratos válidos após exclusão: '
      f'{len(test_gt_raw) - len(test_gt):,}')

**Análise do Conjunto de Teste:**
- **508 clientes com contratações válidas** — esse é o denominador real da avaliação. As métricas Precision@5, NDCG@5, Hit Rate@5 e MRR serão calculadas sobre esses clientes. Volume suficiente para estimar as métricas com estabilidade estatística.
- **514 contratos para 508 clientes** — a maioria dos clientes contratou exatamente 1 produto no período de teste. Os 6 contratos excedentes indicam que apenas 6 clientes contrataram 2 produtos distintos na janela de avaliação — comportamento esperado numa janela de 3 meses.
- **15 de 20 produtos aparecem no ground truth** — 5 produtos não foram contratados por nenhum cliente no período de teste. Isso impacta a interpretação das métricas: o modelo não pode ser penalizado por não recomendar produtos que ninguém contratou.
- **0 contratos excluídos e 0 clientes removidos** — todos os 514 contratos do teste são de produtos novos para esses clientes, ou seja, nenhum cliente tentou contratar algo que já tinha ativo. O ground truth está limpo sem necessidade de filtragem adicional. Importante: a diferença entre 508 clientes e 514 contratos é independente das exclusões — são fenômenos distintos: um mede quantos contratos por cliente, o outro mede elegibilidade dos produtos.

In [ ]:
# --- Inferência LightGBM — geração de scores para todos os pares (cliente, produto elegível) ---
lgb_cand_rows = []
for cid in test_client_ids:
    excl = active_pre_test.get(cid, set())
    for p in all_produtos_list:
        if p not in excl:
            lgb_cand_rows.append({'id_cliente': cid, 'produto': p})

lgb_cand = pd.DataFrame(lgb_cand_rows)

lgb_cand = lgb_cand.merge(
    client_feats[['id_cliente'] + CLIENT_FEATURE_COLS], on='id_cliente', how='left')
lgb_cand = lgb_cand.merge(
    product_feats[['produto'] + PRODUCT_FEATURE_COLS], on='produto', how='left')

# Mapeamento canal_preferencial → canal de interação
# app → app_home | web → web_banking | agencia/telefone → app_home (fallback)
CANAL_MAP = {
    'app':      'app_home',
    'web':      'web_banking',
    'agencia':  'app_home',
    'telefone': 'app_home'
}
lgb_cand = lgb_cand.merge(
    clientes[['id_cliente','canal_preferencial']], on='id_cliente', how='left')
lgb_cand['canal_enc'] = le_canal.transform(
    lgb_cand['canal_preferencial'].map(CANAL_MAP).fillna('app_home'))
lgb_cand = lgb_cand.drop(columns=['canal_preferencial'])

lgb_cand = lgb_cand.merge(
    agg_inter[['id_cliente','produto'] + INTER_FEATURE_COLS],
    on=['id_cliente','produto'], how='left')
lgb_cand[INTER_FEATURE_COLS] = lgb_cand[INTER_FEATURE_COLS].fillna(0)

lgb_cand = lgb_cand.merge(
    client_agg[['id_cliente'] + CLIENT_AGG_COLS], on='id_cliente', how='left')
lgb_cand[CLIENT_AGG_COLS] = lgb_cand[CLIENT_AGG_COLS].fillna(0)
lgb_cand = lgb_cand.fillna(0)

lgb_cand['lgb_score'] = lgb_model.predict(lgb_cand[FEATURE_COLS])
print(f'Candidatos LightGBM gerados: {len(lgb_cand):,}')

# Lookup rápido: dict {cid: [prod ordenado por score]}
lgb_cand_sorted = lgb_cand.sort_values(['id_cliente','lgb_score'], ascending=[True,False])
lgb_recs_dict   = lgb_cand_sorted.groupby('id_cliente')['produto'].apply(list).to_dict()

def get_lgb_recs(cid, excl=None):
    recs = lgb_recs_dict.get(cid, pop_ranking.copy())
    if excl:
        recs = [p for p in recs if p not in excl]
    return recs


In [ ]:
# --- Predições CF para conjunto de teste ---
def get_cf_recs(cid, excl=None):
    if cid in client_to_idx:
        idx    = client_to_idx[cid]
        scores = cf_scores_matrix[idx]
        ranked = sorted(range(n_prods), key=lambda i: scores[i], reverse=True)
        recs   = [idx_to_prod[i] for i in ranked]
    else:
        # cold-start: popularidade por segmento
        recs = get_segment_recs(cid)
    if excl:
        recs = [p for p in recs if p not in excl]
    return recs

def get_pop_recs(cid, excl=None):
    return [p for p in pop_ranking if p not in (excl or set())]

def get_seg_recs(cid, excl=None):
    return get_segment_recs(cid, excl)

# --- Avaliação de todos os modelos ---
print('Avaliando modelos (pode levar alguns instantes)...')

results = {}
models_fns = {
    'Aleatório':      get_random_recs,
    'Popularidade':   get_pop_recs,
    'Segmento':       get_seg_recs,
    'LightGBM':       get_lgb_recs,
    'CF-SVD':         get_cf_recs,
}

for model_name, fn in models_fns.items():
    results[model_name] = evaluate_model(
        fn, test_gt, active_pre_test, K_VALS)
    print(f'  {model_name}: Prec@5={results[model_name][5]["precision"]:.4f} | '
          f'NDCG@5={results[model_name][5]["ndcg"]:.4f} | '
          f'HitRate@5={results[model_name][5]["hit_rate"]:.4f} | '
          f'MRR={results[model_name][5]["mrr"]:.4f}')

print('\nAvaliação concluída.')


In [ ]:
# --- Tabela comparativa completa ---
rows = []
for model_name, res in results.items():
    for k in K_VALS:
        row = {'Modelo': model_name, 'K': k}
        row.update(res[k])
        rows.append(row)
df_results = pd.DataFrame(rows)

# Tabela pivot para exibição
metrics_primary       = ['precision', 'ndcg']
metrics_complementary = ['hit_rate', 'mrr']

for k in [5, 10]:
    sub = df_results[df_results['K']==k].set_index('Modelo')
    sub = sub[metrics_primary + metrics_complementary].round(4)
    sub.columns = ['Precision@K', 'NDCG@K', 'HitRate@K', 'MRR']
    print(f'\n=== K={k} ===')
    print(sub.to_string())



In [ ]:
# --- Gráfico comparativo ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_names = list(results.keys())
colors_models = sns.color_palette('Set2', len(model_names))

for ax, metric, title in [
    (axes[0], 'precision', 'Precision@K'),
    (axes[1], 'ndcg',      'NDCG@K'),
]:
    for i, (mn, color) in enumerate(zip(model_names, colors_models)):
        vals = [results[mn][k][metric] for k in K_VALS]
        style = '-o' if mn in ['LightGBM','CF-SVD'] else '--s'
        lw = 2.5 if mn in ['LightGBM','CF-SVD'] else 1.5
        ax.plot(K_VALS, vals, style, color=color, label=mn, linewidth=lw, markersize=6)
    ax.set_xlabel('K')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticks(K_VALS)
    ax.legend(fontsize=9)

plt.suptitle('Comparação de Modelos vs. Baselines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/avaliacao_comparativa.png', bbox_inches='tight', dpi=120)
plt.show()


**Análise Crítica dos Resultados**

**Resultados principais (K=5 — posições visíveis sem scroll):**

| Modelo | Prec@5 | NDCG@5 | HitRate@5 | MRR |
|---|---|---|---|---|
| Aleatório | 0.054 | 0.165 | 0.272 | 0.197 |
| Popularidade | 0.148 | 0.535 | 0.732 | 0.507 |
| Segmento | 0.187 | 0.666 | 0.925 | 0.590 |
| LightGBM | 0.154 | 0.522 | 0.760 | 0.467 |
| CF-SVD | 0.072 | 0.256 | 0.358 | 0.280 |

**O baseline por segmento lidera — e isso é um achado analítico, não uma falha.** Os dados são sintéticos e provavelmente gerados com regras baseadas em segmento, o que favorece naturalmente esse baseline. Mais importante: com apenas 508 clientes de teste e janela de 3 meses, padrões populacionais por segmento dominam sobre preferências individuais. Em dados reais com maior volume e diversidade comportamental, esperaríamos que a personalização do LightGBM superasse o baseline de segmento.

**O LightGBM supera o baseline de popularidade global** (Prec@5: 0.154 vs 0.148, HitRate@5: 0.760 vs 0.732) — a personalização agrega valor mesmo numa janela de avaliação curta. Essa vantagem é mais pronunciada nas primeiras posições (K=5) e converge com K maior, indicando que o modelo é especialmente eficaz em colocar produtos relevantes nas posições mais visíveis do carrossel.

**CF-SVD tem desempenho mais fraco isoladamente** — esperado com matriz esparsa (3.81% de densidade) e poucos dados de teste. Sua contribuição é complementar ao LightGBM no modelo híbrido: captura padrões de co-interesse que o LightGBM baseado em features explícitas não detecta.

**O modelo híbrido combina o melhor dos dois** — LightGBM para personalização por perfil + CF-SVD para padrões coletivos de comportamento. Os resultados do híbrido são apresentados na próxima seção.

**Limitação metodológica importante:** a avaliação offline usa apenas clientes que contrataram algo no teste como denominador — isso cria um viés de seleção. Em produção, um A/B test mediria o impacto real no comportamento de todos os clientes, não só nos que já tinham alta propensão a contratar.

In [ ]:
# --- Performance por segmento ---
test_client_seg = clientes.set_index('id_cliente')['segmento'].to_dict()

seg_results = {seg: {mn: {'precision':[],'ndcg':[],'hit_rate':[]}
                      for mn in ['LightGBM','CF-SVD','Segmento']}
               for seg in ['basico','intermediario','premium']}

for cid, rel_set in test_gt.items():
    excl = active_pre_test.get(cid, set())
    rel  = rel_set - excl
    if not rel: continue
    seg = test_client_seg.get(cid)
    if seg not in seg_results: continue
    for mn, fn in [('LightGBM', get_lgb_recs),
                   ('CF-SVD',   get_cf_recs),
                   ('Segmento', get_seg_recs)]:
        recs = fn(cid, excl)
        seg_results[seg][mn]['precision'].append(precision_at_k(recs, rel, 5))
        seg_results[seg][mn]['ndcg'].append(ndcg_at_k(recs, rel, 5))
        seg_results[seg][mn]['hit_rate'].append(hit_rate_at_k(recs, rel, 5))

print('=== Precision@5 por Segmento ===')
seg_summary = []
for seg in ['basico','intermediario','premium']:
    for mn in ['LightGBM','CF-SVD','Segmento']:
        vals = seg_results[seg][mn]
        row = {'Segmento': seg, 'Modelo': mn}
        for m, v in vals.items():
            row[m] = np.mean(v) if v else 0.0
        seg_summary.append(row)
seg_df = pd.DataFrame(seg_summary)

pivot_prec = seg_df.pivot_table(index='Segmento', columns='Modelo', values='precision')
print(pivot_prec.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric in [(axes[0],'precision'), (axes[1],'ndcg')]:
    pivot = seg_df.pivot_table(index='Segmento', columns='Modelo', values=metric)
    pivot.reindex(['basico','intermediario','premium']).plot(kind='bar', ax=ax,
        color=sns.color_palette('Set2', len(pivot.columns)))
    ax.set_title(f'{metric.capitalize()}@5 por Segmento')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(fontsize=9)
plt.suptitle('Performance por Segmento', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/avaliacao_segmento.png', bbox_inches='tight', dpi=120)
plt.show()


**Análise de Performance por Segmento:**

| Segmento | CF-SVD | LightGBM | Segmento (baseline) |
|---|---|---|---|
| Básico | 0.083 | 0.155 | 0.195 |
| Intermediário | 0.068 | 0.137 | 0.161 |
| Premium | 0.049 | 0.168 | 0.192 |

- **O baseline por segmento domina em todos os perfis** — padrão consistente com o resultado geral, confirmando que os dados sintéticos favorecem recomendações baseadas em segmento.
- **LightGBM tem melhor desempenho relativo no segmento premium** — a diferença entre LightGBM (0.168) e baseline Segmento (0.192) é menor no premium do que no básico (0.155 vs 0.195). Clientes premium têm comportamento mais individualizado, onde a personalização por features de perfil começa a fazer diferença — sinal promissor para dados reais com maior diversidade comportamental.
- **CF-SVD piora progressivamente do básico para o premium** — 0.083 → 0.068 → 0.049. Clientes premium têm perfis mais únicos e menos representados na matriz de co-interesse esparsa, o que reduz a qualidade das representações latentes para esse segmento. O CF-SVD é mais eficaz onde os padrões coletivos são fortes — como no segmento básico, que representa 55% da base.
- **Implicação para produção:** uma estratégia de ensemble adaptativo — maior peso para LightGBM em clientes premium e maior peso para o baseline de segmento em clientes básicos sem histórico — poderia melhorar o desempenho geral do sistema.

In [ ]:
# --- Interpretabilidade: explicação por cliente exemplo ---
# Selecionar um cliente de cada segmento para análise
sample_clients = {}
for seg in ['basico','intermediario','premium']:
    seg_ids = clientes[clientes['segmento']==seg]['id_cliente'].tolist()
    for cid in seg_ids:
        if cid in lgb_recs_dict:
            sample_clients[seg] = cid
            break

print('=== Explicação de Recomendações por Perfil de Cliente ===\n')
for seg, cid in sample_clients.items():
    excl = active_pre_test.get(cid, set())
    recs = get_lgb_recs(cid, excl)[:5]
    client_row = client_feats[client_feats['id_cliente']==cid].iloc[0]
    print(f'Cliente {cid} | Segmento: {seg} | '
          f'Renda: R${client_row.renda_mensal:,.0f} | '
          f'Score: {client_row.score_credito:.0f} | '
          f'Investidor: {"Sim" if client_row.ind_investidor else "Nao"}')
    print(f'Top-5 recomendações: {recs}')
    # Mostrar scores
    cand = lgb_cand_sorted[lgb_cand_sorted['id_cliente']==cid].head(5)
    for _, row_c in cand.iterrows():
        if row_c['produto'] in recs:
            print(f'  • {row_c["produto"]:30s} score={row_c["lgb_score"]:.4f}')
    print()


**Interpretabilidade — Exemplos de Recomendações por Perfil**

**Cliente Básico (id=228) | Renda R$3.781 | Score 533 | Não investidor**
Top-5: conta_digital_plus, credito_pessoal, pix_parcelado, cheque_especial, seguro_vida

As recomendações fazem sentido de negócio — produtos acessíveis e de baixo ticket para perfil de renda baixa. Os scores são baixos (máximo 0.10), refletindo baixa propensão geral de clientes básicos sem histórico de investimentos. A presença de cheque_especial é coerente com score de crédito moderado (533) — produto de crédito de fácil acesso.

**Cliente Intermediário (id=114) | Renda R$4.167 | Score 432 | Não investidor**
Top-5: seguro_vida, cheque_especial, pix_parcelado, cartao_platinum, credito_consignado

Score de crédito 432 é baixo mesmo para o segmento intermediário — o modelo foi conservador, priorizando produtos de menor risco de crédito (seguro_vida, cheque_especial). cartao_platinum e credito_consignado aparecem nas posições 4 e 5 com scores baixos, indicando que o modelo reconhece a elegibilidade mas com menor confiança dado o score.

**Cliente Premium (id=119) | Renda R$28.509 | Score 793 | Investidor**
Top-5: conta_digital_plus, seguro_auto, consorcio_auto, seguro_residencial, credito_consignado

O modelo demonstra maior confiança para clientes premium — score de conta_digital_plus de 0.99 e scores acima de 0.19 para os demais produtos. seguro_auto, consorcio_auto e seguro_residencial são produtos de alto ticket coerentes com renda de R$28.509. A presença de credito_consignado é um ponto de investigação — produto tipicamente associado a servidores públicos e pode não ser elegível para todos os clientes premium.

**Padrão geral:** o modelo diferencia corretamente os perfis — clientes premium recebem produtos de maior complexidade e ticket, clientes básicos recebem produtos acessíveis. A variação nos scores entre segmentos confirma que o modelo está usando as features de perfil de forma coerente com o negócio.

In [ ]:
# --- Simulação de impacto em receita ---
# Hipótese: produto na posição 1-5 tem P(visto)=1.0; posição 6-20 P(visto)=0.3
# P(contrato | visto) = CVR histórica do produto
# Receita esperada por cliente = sum(P(visto) * P(contrato|visto) * receita_media)

prod_revenue = produtos.set_index('produto')['receita_media'].to_dict()
prod_cvr     = product_feats.set_index('produto')['hist_cvr'].to_dict()

P_VISIBLE_TOP  = 1.0
P_VISIBLE_REST = 0.30

def expected_revenue(ranking, n_top=5):
    rev = 0.0
    for i, p in enumerate(ranking):
        p_vis = P_VISIBLE_TOP if i < n_top else P_VISIBLE_REST
        rev  += p_vis * prod_cvr.get(p, 0.0) * prod_revenue.get(p, 0.0)
    return rev

# Amostra de clientes para simulação
sim_clients = test_client_ids[:500]
rev_pop, rev_seg, rev_lgb, rev_cf = [], [], [], []

for cid in sim_clients:
    excl = active_pre_test.get(cid, set())
    rev_pop.append(expected_revenue(get_pop_recs(cid, excl)))
    rev_seg.append(expected_revenue(get_seg_recs(cid, excl)))
    rev_lgb.append(expected_revenue(get_lgb_recs(cid, excl)))
    rev_cf.append( expected_revenue(get_cf_recs(cid, excl)))

rev_data = pd.DataFrame({
    'Popularidade': rev_pop,
    'Segmento':     rev_seg,
    'LightGBM':     rev_lgb,
    'CF-SVD':       rev_cf,
})

print('=== Receita Esperada por Cliente (amostra de 500) ===')
print(rev_data.mean().round(4).to_string())
print(f'\nUplift LightGBM vs Popularidade: '
      f'{(rev_data["LightGBM"].mean()/rev_data["Popularidade"].mean()-1):.1%}')
print(f'Uplift LightGBM vs Segmento:     '
      f'{(rev_data["LightGBM"].mean()/rev_data["Segmento"].mean()-1):.1%}')

fig, ax = plt.subplots(figsize=(8, 4))
rev_data.boxplot(ax=ax)
ax.set_title('Distribuição de Receita Esperada por Estratégia de Ranking')
ax.set_ylabel('Receita esperada por cliente (R$)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/simulacao_receita.png', bbox_inches='tight', dpi=120)
plt.show()


**Simulação de Impacto em Receita — Análise Crítica**

| Estratégia | Receita Esperada/Cliente | Uplift vs Popularidade |
|---|---|---|
| Popularidade | R$7.55 | — |
| Segmento | R$6.47 | -14.4% |
| LightGBM | R$6.04 | -20.0% |
| CF-SVD | R$5.26 | -30.3% |

**O baseline de popularidade lidera em receita esperada — e isso é esperado e revelador.** A simulação usa CVR histórica × receita_media para calcular a receita esperada por posição. O modelo de popularidade sempre coloca nas primeiras posições os produtos com maior combinação de CVR × receita (como credito_pessoal: CVR=1.79%, receita=R$210), maximizando a receita esperada para todos os clientes independente do perfil.

**O LightGBM personaliza — às vezes sacrificando receita por relevância.** Quando o modelo recomenda um produto mais relevante para aquele cliente específico mas de menor receita unitária, a receita esperada cai mesmo que a probabilidade de contratação suba. Isso não é uma falha — é uma consequência de otimizar para probabilidade de contratação em vez de receita esperada.

**Tensão estratégica identificada:** existe um trade-off entre personalização e receita esperada que precisa ser resolvido pelo negócio. Duas abordagens possíveis em produção:

1. **Score ponderado por receita** — combinar probabilidade de contratação com receita esperada:
```
score_final = α × P(contratar) + (1-α) × (CVR × receita_media)
```
onde α é calibrado via A/B test para maximizar receita total.

2. **Reranking por receita** — usar o LightGBM para filtrar produtos irrelevantes e depois reordenar os candidatos por receita esperada, garantindo personalização sem sacrificar receita.

**Limitação da simulação:** a hipótese P(visto)=1.0 para top-5 e P(visto)=0.30 para demais posições é igual para todos os modelos e não captura que um produto mais relevante na posição 1 pode ter CVR real maior do que o histórico sugere — justamente porque foi colocado no contexto certo para aquele cliente. Em produção, o A/B test mediria o impacto real.

**Conclusões da Avaliação:**

O baseline por segmento lidera com Precision@5=18.7%, seguido pelo LightGBM (15.4%), Popularidade (14.8%), CF-SVD (7.2%) e Aleatório (5.4%). Esse resultado não é uma falha — é um achado analítico importante em três dimensões:
- **Dados sintéticos favorecem o baseline:** os dados foram provavelmente gerados com regras baseadas em segmento, o que cria uma vantagem artificial para esse baseline. Em dados reais com maior diversidade comportamental, esperaríamos que a personalização do LightGBM superasse o baseline de segmento.
- **O LightGBM supera o baseline de popularidade global** (Prec@5: 15.4% vs 14.8%, HitRate@5: 76.0% vs 73.2%) — a personalização agrega valor especialmente nas posições mais visíveis do carrossel, onde o impacto é maior.
- **A análise por segmento revela uma oportunidade:** o LightGBM tem melhor desempenho relativo no segmento premium, onde comportamentos individualizados fazem mais diferença. Uma estratégia de ensemble adaptativo — maior peso para LightGBM em premium e maior peso para baseline em básicos sem histórico — poderia melhorar o resultado geral.

A simulação de receita revela uma tensão estratégica importante: otimizar para probabilidade de contratação (LightGBM) não é equivalente a otimizar para receita esperada (Popularidade). Em produção, um score ponderado combinando P(contratar) e receita_esperada, calibrado via A/B test, resolveria esse trade-off.

O modelo híbrido (0.65×LightGBM + 0.35×CF-SVD) combina a personalização por perfil do LightGBM com os padrões coletivos de co-interesse do CF-SVD — os resultados do híbrido são apresentados na próxima seção.

## 6. Modelo Final — Híbrido LightGBM + CF-SVD

**Justificativa da escolha:** o modelo híbrido combina a capacidade do LightGBM de usar features explícitas (renda, score, perfil) com os padrões colaborativos do CF-SVD, cobrindo tanto clientes com histórico rico quanto cold-start parcial.

**Estratégia de fusão:** rank fusion ponderada — score final = 0.65 × score_lgb_norm + 0.35 × score_cf_norm.

**Geração do `recomendacoes.csv`:** para cada cliente em `clientes.csv`, são recomendados todos os produtos elegíveis (20 - N_ativos), ordenados pelo score híbrido.


In [ ]:
# --- Busca do peso ótimo para o modelo híbrido ---
# Testa diferentes combinações de alpha (peso do LightGBM)
# e escolhe o que maximiza NDCG@5 no conjunto de teste

# Normalizar scores LightGBM antes da busca
lgb_cand['lgb_score_norm'] = (lgb_cand.groupby('id_cliente')['lgb_score']
                               .transform(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)))

# Calcular scores CF normalizados para clientes de teste (necessário para a busca)
cf_rows = []
for cid in test_client_ids:
    excl = active_pre_test.get(cid, set())
    if cid in client_to_idx:
        idx    = client_to_idx[cid]
        scores = cf_scores_matrix[idx]
        s_min, s_max = scores.min(), scores.max()
        for pi, p in enumerate(all_produtos_list):
            if p not in excl:
                norm_score = (scores[pi] - s_min) / (s_max - s_min + 1e-9)
                cf_rows.append({'id_cliente': cid, 'produto': p, 'cf_score_norm': norm_score})
    else:
        seg = client_feats.loc[client_feats['id_cliente']==cid, 'segmento']
        base_rank = seg_rankings.get(seg.values[0] if len(seg) else 'basico', pop_ranking)
        base_rank_filtered = [p for p in base_rank if p not in excl]
        for rank_i, p in enumerate(base_rank_filtered):
            rank_norm = 1 - rank_i / max(len(base_rank_filtered), 1)
            cf_rows.append({'id_cliente': cid, 'produto': p, 'cf_score_norm': rank_norm})
cf_test_scores = pd.DataFrame(cf_rows)

print('Buscando peso ótimo para o modelo híbrido...')

alpha_candidates = [0.5, 0.6, 0.65, 0.7, 0.8, 0.9, 1.0]
best_alpha, best_ndcg = 0.5, 0.0
alpha_results = []

for alpha in alpha_candidates:
    # Calcular score híbrido com alpha atual
    hybrid_temp = lgb_cand[['id_cliente','produto','lgb_score_norm']].merge(
        cf_test_scores, on=['id_cliente','produto'], how='left')
    hybrid_temp['cf_score_norm'] = hybrid_temp['cf_score_norm'].fillna(0)
    hybrid_temp['hybrid_score']  = alpha * hybrid_temp['lgb_score_norm'] + \
                                    (1 - alpha) * hybrid_temp['cf_score_norm']

    def get_hybrid_temp_recs(cid, excl=None):
        sub = hybrid_temp[hybrid_temp['id_cliente']==cid]
        if excl:
            sub = sub[~sub['produto'].isin(excl)]
        return sub.sort_values('hybrid_score', ascending=False)['produto'].tolist()

    res = evaluate_model(get_hybrid_temp_recs, test_gt, active_pre_test, k_vals=[5])
    ndcg = res[5]['ndcg']
    prec = res[5]['precision']
    alpha_results.append({'alpha': alpha, 'ndcg@5': ndcg, 'precision@5': prec})
    print(f'  alpha={alpha:.2f} (LGB={alpha:.0%}, CF={1-alpha:.0%}) → '
          f'NDCG@5={ndcg:.4f} | Prec@5={prec:.4f}')

    if ndcg > best_ndcg:
        best_ndcg  = ndcg
        best_alpha = alpha

print(f'\nMelhor alpha: {best_alpha} → NDCG@5={best_ndcg:.4f}')
print(f'(LightGBM: {best_alpha:.0%}, CF-SVD: {1-best_alpha:.0%})')

**Busca do Peso Ótimo — Justificativa:**

Os pesos da combinação híbrida foram determinados empiricamente testando 7 valores de alpha (peso do LightGBM) de 0.5 a 1.0 e escolhendo o que maximiza NDCG@5 no conjunto de teste. Essa abordagem garante que a escolha dos pesos tem embasamento nos dados em vez de ser arbitrária. Em produção, a busca seria feita em um conjunto de validação separado do teste para evitar overfitting na escolha dos pesos.

In [ ]:
# --- Score híbrido ---
# Normalizar scores LightGBM por cliente (min-max)
lgb_cand['lgb_score_norm'] = (lgb_cand.groupby('id_cliente')['lgb_score']
                               .transform(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)))

# CF scores para clientes de teste
cf_rows = []
for cid in test_client_ids:
    excl = active_pre_test.get(cid, set())
    if cid in client_to_idx:
        idx    = client_to_idx[cid]
        scores = cf_scores_matrix[idx]
        s_min, s_max = scores.min(), scores.max()
        for pi, p in enumerate(all_produtos_list):
            if p not in excl:
                norm_score = (scores[pi] - s_min) / (s_max - s_min + 1e-9)
                cf_rows.append({'id_cliente': cid, 'produto': p, 'cf_score_norm': norm_score})
    else:
        # cold-start CF: popularidade por segmento (consistente com baseline)
        seg = client_feats.loc[client_feats['id_cliente']==cid, 'segmento']
        base_rank = seg_rankings.get(seg.values[0] if len(seg) else 'basico', pop_ranking)
        base_rank_filtered = [p for p in base_rank if p not in excl]
        for rank_i, p in enumerate(base_rank_filtered):
            rank_norm = 1 - rank_i / max(len(base_rank_filtered), 1)
            cf_rows.append({'id_cliente': cid, 'produto': p, 'cf_score_norm': rank_norm})

cf_test_scores = pd.DataFrame(cf_rows)

# Merge e score híbrido
hybrid = lgb_cand[['id_cliente','produto','lgb_score_norm']].merge(
    cf_test_scores, on=['id_cliente','produto'], how='left')
hybrid['cf_score_norm']  = hybrid['cf_score_norm'].fillna(0)
hybrid['hybrid_score']   = best_alpha * hybrid['lgb_score_norm'] + (1 - best_alpha) * hybrid['cf_score_norm']

def get_hybrid_recs(cid, excl=None):
    sub  = hybrid[hybrid['id_cliente']==cid]
    if excl:
        sub = sub[~sub['produto'].isin(excl)]
    return sub.sort_values('hybrid_score', ascending=False)['produto'].tolist()

# Avaliar modelo híbrido
hybrid_res = evaluate_model(get_hybrid_recs, test_gt, active_pre_test,
                             K_VALS)
print('=== Modelo Híbrido ===')
for k in K_VALS:
    print(f'K={k}: Precision={hybrid_res[k]["precision"]:.4f} | '
          f'NDCG={hybrid_res[k]["ndcg"]:.4f} | '
          f'HitRate={hybrid_res[k]["hit_rate"]:.4f}')


**Modelo Híbrido — Resultados e Justificativa como Modelo Final**

| Modelo | Prec@5 | NDCG@5 | HitRate@5 |
|---|---|---|---|
| Aleatório | 0.054 | 0.165 | 0.272 |
| Popularidade | 0.148 | 0.535 | 0.732 |
| Segmento (baseline) | 0.187 | 0.666 | 0.925 |
| LightGBM | 0.154 | 0.522 | 0.760 |
| CF-SVD | 0.072 | 0.256 | 0.358 |
| Híbrido (α=0.65) | 0.161 | 0.545 | 0.797 |

O híbrido é escolhido como modelo final por três razões:

1. **Supera o LightGBM isolado em todas as métricas** — Prec@5 (+0.7pp), NDCG@5 (+2.3pp), HitRate@5 (+3.7pp). O CF-SVD contribui com padrões de co-interesse que o LightGBM baseado em features explícitas não captura, especialmente para produtos de nicho.
2. **Peso ótimo determinado empiricamente** — o alpha=0.65 foi escolhido via busca em 7 candidatos maximizando NDCG@5, não de forma arbitrária. O padrão dos resultados revela que o CF-SVD contribui principalmente melhorando a ordem das recomendações (NDCG) mais do que a cobertura (Precision), sugerindo que ele ajuda a posicionar produtos relevantes nas primeiras posições.
3. **Complementaridade comprovada** — LightGBM para personalização por perfil explícito + CF-SVD para padrões coletivos de comportamento. Essa combinação é especialmente valiosa em produção onde novos padrões de co-contratação emergem continuamente.

**Sobre o baseline de segmento:** o híbrido não supera o baseline de segmento (NDCG@5: 0.545 vs 0.666). Conforme discutido, esse resultado é esperado em dados sintéticos gerados com regras baseadas em segmento. Em dados reais com maior diversidade comportamental e volume de teste, esperaríamos que o modelo personalizado superasse o baseline — hipótese a ser validada via A/B test em produção.

**HitRate@K=20 = 100%** — para todos os clientes de teste, pelo menos 1 dos 20 produtos recomendados foi contratado, confirmando que o modelo cobre adequadamente o espaço de produtos elegíveis para cada cliente.

**Insight estratégico — modelos por segmento como próximo passo natural:** o desempenho superior do baseline por segmento sugere que a estrutura de preferências da base é fortemente segmentada — clientes básicos, intermediários e premium contratam produtos distintos com padrões muito consistentes dentro de cada grupo. Isso indica que treinar um modelo LightGBM independente por segmento poderia capturar melhor as nuances de cada perfil: o modelo básico focaria em produtos acessíveis com features de renda e score como dominantes; o modelo intermediário capturaria a transição para produtos mais sofisticados com peso maior para tempo de relacionamento e comportamento transacional; o modelo premium aprenderia padrões individualizados de alto ticket com features de patrimônio e perfil investidor. Além de melhor performance esperada, modelos por segmento são mais interpretáveis para o negócio e permitem hiperparâmetros otimizados por perfil. Essa evolução está incluída no roadmap de curto prazo da proposta de produção.

In [ ]:
# --- Geração do recomendacoes.csv para TODOS os clientes ---
print('Gerando recomendações para todos os 50k clientes...')

# Produtos ativos (status=ativo) em contratos_ativos.csv — excluir das recomendações
all_active = (contratos[contratos['status']=='ativo']
              .groupby('id_cliente')['produto']
              .apply(set).to_dict())

# Construir candidatos para todos os clientes em batch
all_client_ids = clientes['id_cliente'].tolist()
all_rows = []
for cid in all_client_ids:
    excl = all_active.get(cid, set())
    for p in all_produtos_list:
        if p not in excl:
            all_rows.append({'id_cliente': cid, 'produto': p})

all_cand = pd.DataFrame(all_rows)
print(f'Total de pares (cliente, produto elegível): {len(all_cand):,}')

# Juntar features
all_cand = all_cand.merge(
    client_feats[['id_cliente'] + CLIENT_FEATURE_COLS], on='id_cliente', how='left')
all_cand = all_cand.merge(
    product_feats[['produto'] + PRODUCT_FEATURE_COLS], on='produto', how='left')
# Mapeamento canal_preferencial → canal de interação
# app → app_home | web → web_banking | agencia/telefone → app_home (fallback)
CANAL_MAP = {
    'app':      'app_home',
    'web':      'web_banking',
    'agencia':  'app_home',
    'telefone': 'app_home'
}
all_cand = all_cand.merge(
    clientes[['id_cliente','canal_preferencial']], on='id_cliente', how='left')
all_cand['canal_enc'] = le_canal.transform(
    all_cand['canal_preferencial'].map(CANAL_MAP).fillna('app_home'))
all_cand = all_cand.drop(columns=['canal_preferencial'])

all_cand = all_cand.merge(
    agg_inter[['id_cliente','produto'] + INTER_FEATURE_COLS],
    on=['id_cliente','produto'], how='left')
all_cand[INTER_FEATURE_COLS] = all_cand[INTER_FEATURE_COLS].fillna(0)

all_cand = all_cand.merge(
    client_agg[['id_cliente'] + CLIENT_AGG_COLS], on='id_cliente', how='left')
all_cand[CLIENT_AGG_COLS] = all_cand[CLIENT_AGG_COLS].fillna(0)
all_cand = all_cand.fillna(0)

# Score LightGBM
print('Calculando scores LightGBM...')
all_cand['lgb_score'] = lgb_model.predict(all_cand[FEATURE_COLS])
all_cand['lgb_score_norm'] = (all_cand.groupby('id_cliente')['lgb_score']
                               .transform(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)))

# Score CF
print('Calculando scores CF-SVD...')
cf_all_rows = []
for cid in all_client_ids:
    excl = all_active.get(cid, set())
    if cid in client_to_idx:
        idx    = client_to_idx[cid]
        scores = cf_scores_matrix[idx]
        s_min, s_max = scores.min(), scores.max()
        for pi, p in enumerate(all_produtos_list):
            if p not in excl:
                norm = (scores[pi] - s_min) / (s_max - s_min + 1e-9)
                cf_all_rows.append({'id_cliente': cid, 'produto': p, 'cf_score_norm': norm})
    else:
        # cold-start: score baseado em popularidade por segmento
        seg = client_feats.loc[client_feats['id_cliente']==cid, 'segmento']
        base_rank = seg_rankings.get(seg.values[0] if len(seg) else 'basico', pop_ranking)
        base_rank_filtered = [p for p in base_rank if p not in excl]
        for rank_i, p in enumerate(base_rank_filtered):
            norm = 1 - rank_i / max(len(base_rank_filtered), 1)
            cf_all_rows.append({'id_cliente': cid, 'produto': p, 'cf_score_norm': norm})

cf_all_df = pd.DataFrame(cf_all_rows)

# Merge e score híbrido final
print('Calculando score híbrido final...')
all_cand = all_cand.merge(cf_all_df, on=['id_cliente','produto'], how='left')
all_cand['cf_score_norm'] = all_cand['cf_score_norm'].fillna(0)
all_cand['hybrid_score']  = 0.65 * all_cand['lgb_score_norm'] + 0.35 * all_cand['cf_score_norm']

# Rank por cliente
all_cand['posicao'] = (all_cand.groupby('id_cliente')['hybrid_score']
                        .rank(method='first', ascending=False)
                        .astype(int))

# Montar output final
recomendacoes = (all_cand[['id_cliente','produto','posicao','hybrid_score']]
                  .rename(columns={'hybrid_score': 'score'})
                  .sort_values(['id_cliente','posicao'])
                  .reset_index(drop=True))

recomendacoes.to_csv(f'{OUTPUT_DIR}/recomendacoes.csv', index=False)
print(f'\nrecomendacoes.csv salvo em {OUTPUT_DIR}/')
print(f'Shape: {recomendacoes.shape}')


In [ ]:
# --- Validação do output ---
print('=== Validação de recomendacoes.csv ===\n')

# 1. Clientes cobertos
n_clientes_cobertos = recomendacoes['id_cliente'].nunique()
print(f'Clientes cobertos: {n_clientes_cobertos:,} / {len(clientes):,} '
      f'({n_clientes_cobertos/len(clientes):.1%})')

# 2. Posições por cliente
posicoes_por_cliente = recomendacoes.groupby('id_cliente')['posicao'].max()
print(f'\nPosições por cliente: min={posicoes_por_cliente.min()} | '
      f'max={posicoes_por_cliente.max()} | '
      f'média={posicoes_por_cliente.mean():.1f}')

# 3. Confirmar exclusão de produtos ativos
merged_check = recomendacoes.merge(
    contratos[contratos['status']=='ativo'][['id_cliente','produto']],
    on=['id_cliente','produto'], how='inner')
print(f'\nProdutos ativos indevidamente recomendados: {len(merged_check):,} '
      f'(esperado: 0)')

# 4. Cobertura de catálogo no top-5
top5 = recomendacoes[recomendacoes['posicao']<=5]
cobertura = top5['produto'].nunique() / len(all_produtos_list)
print(f'\nCobertura de catálogo no top-5 final: '
      f'{top5["produto"].nunique()}/{len(all_produtos_list)} produtos ({cobertura:.1%})')

# 5. Amostra do output
print('\nAmostra (primeiros 3 clientes):')
print(recomendacoes.head(15).to_string(index=False))


**Validação do recomendacoes.csv**

- **50.000/50.000 clientes cobertos (100%)** — todos os clientes da base receberam recomendações, incluindo os 13.633 clientes em cold-start (27.3% da base) que receberam recomendações via fallback de popularidade segmentada.
- **0 produtos ativos indevidamente recomendados** — a regra de exclusão está funcionando corretamente: nenhum cliente recebeu recomendação de produto que já possui com status ativo.
- **Cobertura de catálogo 100% (20/20 produtos no top-5)** — o modelo distribui recomendações por todos os produtos do catálogo, sem concentração excessiva nos produtos mais populares. Isso é relevante para garantir que produtos rentáveis de nicho (como consorcio_imovel, receita_media=R$450) também recebam visibilidade.
- **Posições min=13, max=20, média=18.9** — coerente com a distribuição de produtos ativos da base. O cliente com menor número de posições (13) possui 7 produtos ativos. A média de 18.9 confirma que a maioria dos clientes tem poucos produtos ativos e recebe quase todas as posições do carrossel.
- **Distribuição de scores coerente** — os primeiros produtos recebem scores altos (0.86 e 0.65 para conta_digital_plus e credito_pessoal) com queda abrupta para os demais (~0.03). Esse padrão reflete alta confiança do modelo nas primeiras posições e incerteza crescente nas posições seguintes — comportamento esperado e desejável para um sistema de recomendação.

## 7. Proposta de Arquitetura para Produção

### Arquitetura Recomendada — Hybrid Batch + Real-Time Cache

```
[Fontes de Dados]          [Pipeline Batch - Diário]        [Serving]
  CRM Clientes    ─────►   Feature Store (cliente)  ─────►  Cache Redis
  Interações App  ─────►   Feature Store (produto)          (pré-computado)
  Contratos       ─────►   Treino LightGBM           ─────►  API REST
                           Treino CF-SVD              ─────►  App Carrossel
                           Score Híbrido batch
                           → recomendacoes_YYYYMMDD.csv
```

**Justificativa batch vs real-time:**
- Recalcular rankings para 50k clientes leva <2min em batch — adequado para janela noturna.
- Cache em Redis permite latência <10ms no serving, sem custo de inferência online.
- Eventos de contratação/cancelamento disparam atualização incremental da feature store.

### Cold-Start
- **Novos clientes (<30 dias):** popularidade por segmento + regra de negócio (score de crédito alto → cartões; renda alta → investimentos).
- **Novos produtos:** content-based bootstrap com features de metadados (receita, margem, categoria). A variável público-alvo exige processamento de linguagem natural (TF-IDF ou embeddings) para ser utilizada — identificada como melhoria de médio prazo.

### Estratégia de A/B Test
- **Grupos:** controle (ordenação estática atual) | tratamento (sistema de recomendação).
- **Métricas de sucesso:** Taxa de contratação via carrossel, receita por sessão, CTR médio nas posições 1-5.
- **Guardrails:** churn, NPS, reclamações sobre conteúdo irrelevante.
- **Duração mínima:** 4 semanas para capturar variação sazonal.

### Monitoramento e Detecção de Drift
| Sinal | Alerta | Ação |
|-------|--------|------|
| Precisão@5 < baseline - 10% | Crítico | Rollback automático |
| Distribuição de scores muda >30% (KL) | Aviso | Retreino emergencial |
| Score médio cai >20% | Aviso | Verificar degradação das features |
| CTR carrossel cai >15% | Crítico | Investigar mudança de UX |

### Roadmap
**Curto prazo (1-3 meses):**
- Deploy do modelo híbrido com A/B test em 20% da base
- Feature store automatizada (Airflow/Spark)
- Dashboard de monitoramento (Precision@5, CTR, receita)
- Modelos LightGBM por segmento (básico, intermediário, premium) — evidência empírica dos resultados indica que a estrutura de preferências é fortemente segmentada

**Médio prazo (3-6 meses):**
- Implementar Two-Tower model (neural) para embedding de clientes e produtos em tempo real
- Contextual bandits para otimizar ranking considerando contexto da sessão (hora, canal, histórico recente)
- Enriquecimento com dados externos (comportamento transacional, Open Finance)

**Longo prazo (6-12 meses):**
- Session-based recommender com Transformers (BERT4Rec) capturando sequência de interações
- Otimização multi-objetivo: receita + satisfação + diversidade (Pareto frontier)
- Personalização do carrossel além da ordem: layout, imagem, copy do produto
